# Diarrhea ionic programs and pediatric vulnerability

**Question (PI-facing).** *Why are young children disproportionately affected by severe diarrhea?* This notebook tests gene-level hypotheses around ionic / channel programs in absorptive epithelium, building on the existing finding that **`SLC26A3` is overwhelmingly expressed in BEST4 absorptive cells** in the HGCA single-cell and Visium spatial data, and that BEST4 cells (Enterocytes + Colonocytes) are heterogeneous along the gut axis.

The analysis follows the same metadata-aware best practices used in the compositions notebooks (`Composition_along_gut_segments.ipynb`, `composition_along_age_ranges.ipynb`):

- **Atlas axis.** `tissue_level_1 ∈ {duodenum, jejunum, ileum, colon}` for the gut axis. Atlas imbalance is acknowledged: jejunum is mostly immune samples, duodenum has only ~9 epithelial-rich samples; analyses are stratified or per-tissue when sensible and otherwise use the atlas-wide model.\n- **Age axis.** `age_range` (decade bins) with a binary pediatric (0–9) vs adult contrast, because the original question is specifically about young children.\n- **Cell-type axis.** `hgca_celltype_v1`, restricted to the absorptive epithelial lineage (BEST4 Enterocytes/Colonocytes, regular Enterocytes/Colonocytes, crypt-top, progenitors).\n- **Statistics.** Pseudobulk to sample × cell type × tissue. Metadata-aware OLS with cubic-spline age effects: `expression ~ bs(age_order, df=3) + tissue_level_1 + assay + sampled_site_condition + sample_collection_method + sample_preservation_method`. Pediatric vs adult contrasts use the same covariate adjustment.\n- **CCC centrality.** Re-uses precomputed LIANA outputs in `LIANA/ccc_centrality_tissue_level_1/` and `LIANA/per_tissue_level_1/HCA_*.csv` to ask whether BEST4 cells are network-central in segments where diarrhea-relevant ligand–receptor pairs (CFTR, GUCA2A/B–GUCY2C, etc.) dominate.\n\n**Hypotheses tested.**\n\n1. **`SLC26A3` is the BEST4 marker** of the absorptive lineage and its dominance is gut-segment-resolved (BEST4 Colonocytes ≥ BEST4 Enterocytes ≫ regular absorptive).\n2. **Pediatric absorptive cells over-express the secretory machinery** (`CFTR`, `GUCY2C`, `KCNN4`, `SLC12A2 / NKCC1`) that diarrheagenic toxins co-opt, **AND/OR under-express the absorptive machinery** (`SLC26A3`, `SLC9A3 / NHE3`, `SLC5A1 / SGLT1`).\n3. **BEST4 cell fraction within absorptive epithelium is higher in young children**, after accounting for tissue and metadata.\n4. **BEST4 is centrally placed in segment-specific cell–cell communication networks** that include ionic-program ligands/receptors.\n\n**Outputs** are written under `~/GCA/github_vignette_output/diarrhea_ionic_programs/`, matching the convention of the BEST4 / compositions notebooks.\n\n**Plot styling** follows `~/Projects/GCA/publication2026/plot_specs.md`: Helvetica, 6 pt body / 7 pt title, **no gridlines**, L-shaped axes, **Wong colorblind-safe palette**, panel sizes that resolve to 90 mm or 180 mm at final size.

## Setup, paper-style helpers, and output directory

Helpers below standardize plot styling per `plot_specs.md` so every figure exports at a final size (90 mm or 180 mm wide) with Helvetica, 6 pt text, and no gridlines. Vector PDF + SVG are written next to each PNG preview so panels are Illustrator-editable.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import os
import re
import json

import numpy as np
import pandas as pd
import scipy.sparse as sp

import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

import scanpy as sc
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests
from patsy import bs
from patsy.builtins import Q

# Wong colorblind-safe palette (plot_specs.md section 9)
WONG = {
    "black":      "#000000",
    "vermillion": "#D55E00",
    "blue":       "#0072B2",  # HCA blue
    "green":      "#009E73",
    "sky":        "#56B4E9",
    "yellow":     "#F0E442",
    "purple":     "#CC79A7",
    "lightgrey":  "#E0E0E0",
    "midgrey":    "#999999",
}

# Project-specific role assignments
SEGMENT_COLORS = {
    "duodenum": WONG["sky"],
    "jejunum":  WONG["green"],
    "ileum":    WONG["yellow"],
    "colon":    WONG["purple"],
}
LINEAGE_COLORS = {
    "BEST4 Enterocytes":         WONG["blue"],
    "BEST4 Colonocytes":         WONG["vermillion"],
    "Villus Tip Enterocytes":    WONG["green"],
    "Mid Villus Enterocytes":    WONG["sky"],
    "Lower Villus Enterocytes":  WONG["yellow"],
    "Enterocyte Progenitors":    WONG["midgrey"],
    "Crypt Top Colonocytes":     WONG["purple"],
    "Mid Crypt Colonocytes":     "#8c5b1f",
    "Lower Crypt Colonocytes":   "#3f5e2a",
    "Colonocyte Progenitors":    "#6c6c6c",
}

# Apply Nature plot specs as matplotlib rcParams. We export at final size.
def apply_paper_style():
    mpl.rcParams.update({
        "font.family": "Helvetica",
        "font.size": 6,
        "axes.labelsize": 6,
        "axes.titlesize": 7,
        "axes.titleweight": "bold",
        "xtick.labelsize": 6,
        "ytick.labelsize": 6,
        "legend.fontsize": 6,
        "legend.title_fontsize": 6,
        "axes.linewidth": 0.5,
        "axes.edgecolor": "black",
        "axes.spines.top":   False,
        "axes.spines.right": False,
        "xtick.major.width": 0.5,
        "ytick.major.width": 0.5,
        "xtick.major.size":  2,
        "ytick.major.size":  2,
        "xtick.direction":   "out",
        "ytick.direction":   "out",
        "axes.grid":         False,
        "savefig.transparent": False,
        "savefig.dpi":       300,
        "pdf.fonttype":      42,   # editable text in Illustrator
        "ps.fonttype":       42,
        "svg.fonttype":      "none",
        "figure.facecolor":  "white",
        "axes.facecolor":    "white",
    })

apply_paper_style()

# Width helpers ---------------------------------------------------------
MM_PER_IN = 25.4
def mm(width_mm, height_mm):
    """Convert mm tuple -> matplotlib (in, in) figsize."""
    return (width_mm / MM_PER_IN, height_mm / MM_PER_IN)

# Save in PDF + SVG + PNG; PNG only as preview, PDF/SVG are deliverables
def save_figure(fig, base, width_mm, height_mm):
    base = Path(str(base))
    base.parent.mkdir(parents=True, exist_ok=True)
    fig.set_size_inches(width_mm / MM_PER_IN, height_mm / MM_PER_IN)
    fig.savefig(base.with_suffix(".pdf"), bbox_inches="tight")
    fig.savefig(base.with_suffix(".svg"), bbox_inches="tight")
    fig.savefig(base.with_suffix(".png"), bbox_inches="tight", dpi=300)


def style_ax(ax, linewidth=0.5):
    ax.grid(False)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    for s in ("left", "bottom"):
        ax.spines[s].set_color("black")
        ax.spines[s].set_linewidth(linewidth)
    ax.tick_params(axis="both", which="both", color="black", width=linewidth, length=2)

OUTDIR = Path("/Users/kylekimler/GCA/github_vignette_output/diarrhea_ionic_programs")
OUTDIR.mkdir(parents=True, exist_ok=True)
print(f"Output dir: {OUTDIR}")

LIANA_DIR = Path("/Users/kylekimler/Projects/GCA/github_vignette_output/LIANA")
print(f"LIANA dir: {LIANA_DIR} (exists={LIANA_DIR.exists()})")

EPITHELIAL_PATH = "/Users/kylekimler/Projects/GCA/meta_datasets/integrated-objects/epithelial.h5ad"

## Diarrhea ionic gene panels by clinical mechanism

These are the gene panels that drive the rest of this notebook. They are organized by **clinical mechanism / disease subtype** rather than by cell type so that the same panel can be applied to every cell type and we can ask *who* expresses *which mechanism* across the gut axis and the lifespan.

| Panel | Mechanism | Diarrhea relevance |
|---|---|---|
| `secretory_cAMP_apical` | cAMP-driven apical Cl⁻ secretion | Cholera toxin, ETEC LT (CFTR-driven) |
| `secretory_cGMP_apical` | cGMP-driven apical Cl⁻ secretion | ETEC ST → GUCY2C → CFTR |
| `secretory_basolateral` | Basolateral Cl⁻ entry / K⁺ recycling | Same outflow path; rate-limiting |
| `na_absorption_apical` | Electroneutral Na⁺ absorption | Loss = secretory phenotype (NHE3 inhibition) |
| `cl_absorption_dra` | Apical Cl⁻ absorption / Cl⁻↔HCO₃⁻ exchange | Congenital chloride diarrhea (`SLC26A3`) |
| `congenital_diarrhea` | Monogenic congenital diarrhea drivers | Rare but illuminating disease genes |
| `aquaporins` | Trans-epithelial water permeability | Net water flux follows ions |
| `sglt_glucose_water` | Na⁺-glucose cotransport | Mechanism behind oral rehydration salts |
| `bile_acid_uptake` | Ileal bile acid reabsorption | Bile-acid diarrhea (BAD) |
| `best4_sensor_program` | BEST4 identity / sensory programs | The cell type at the center of this story |
| `tight_junction_barrier` | Paracellular leak | Inflammatory / barrier-loss diarrhea |
| `inflammasome_secretagogues` | Endogenous secretagogues | Drive secretion in inflammatory diarrhea |

In [ ]:
# Diarrhea ionic gene panels (gene symbols, HGNC). Curated by clinical mechanism.
DIARRHEA_PANELS = {
    "secretory_cAMP_apical": [
        "CFTR",        # apical Cl- channel; PKA-activated
        "CLCA1",       # Ca2+-activated Cl- (mucinous; goblet-leaning)
        "CLCA4",
        "ANO1",        # TMEM16A; Ca2+-activated Cl-
    ],
    "secretory_cGMP_apical": [
        "GUCY2C",      # heat-stable enterotoxin receptor (ETEC ST)
        "GUCA2A",      # endogenous ligand (uroguanylin)
        "GUCA2B",      # endogenous ligand (guanylin)
        "PKG2",        # PRKG2 cGMP-dependent kinase II (apical)
        "PRKG2",
    ],
    "secretory_basolateral": [
        "SLC12A2",     # NKCC1; basolateral Cl- entry for secretion
        "KCNN4",       # IK / SK4; basolateral K+ recycling for secretion
        "KCNQ1",       # cAMP-activated K+ recycling partner
        "KCNE3",
        "ATP1A1",      # Na/K ATPase
        "ATP1B1",
        "SLC4A4",      # NBCe1; basolateral HCO3- entry
    ],
    "na_absorption_apical": [
        "SLC9A3",      # NHE3 apical Na/H
        "SLC9A2",      # NHE2
        "NHERF1",      # SLC9A3R1 scaffold
        "NHERF2",      # SLC9A3R2
        "SLC9A3R1",
        "SLC9A3R2",
    ],
    "cl_absorption_dra": [
        "SLC26A3",     # DRA; apical Cl-/HCO3- exchanger; CCD gene
        "SLC26A6",     # PAT-1; partner exchanger
        "CFTR",        # required for HCO3- efflux side
    ],
    "congenital_diarrhea": [
        "SLC26A3",     # congenital chloride diarrhea
        "SPINT2",      # congenital sodium diarrhea
        "SLC9A3",      # congenital sodium diarrhea (autosomal)
        "GUCY2C",      # MEDNIK / FCSD
        "MYO5B",       # microvillus inclusion disease
        "STX3",        # MVID
        "STXBP2",
        "EPCAM",       # CTE / tufting enteropathy
        "TTC7A",       # multiple intestinal atresia / VEOIBD
        "DGAT1",       # protein-losing enteropathy with diarrhea
        "NEUROG3",     # enteric anendocrinosis
        "PCSK1",       # endocrine cell processing
        "DPP4",        # not causal but pharmacology
    ],
    "aquaporins": [
        "AQP3", "AQP4", "AQP5", "AQP7", "AQP8", "AQP10", "AQP11",
    ],
    "sglt_glucose_water": [
        "SLC5A1",      # SGLT1 - the mechanism behind ORS
        "SLC2A2",      # GLUT2
        "SLC2A5",      # GLUT5 fructose
        "SI",          # sucrase-isomaltase
        "MGAM",        # maltase-glucoamylase
        "LCT",         # lactase
    ],
    "bile_acid_uptake": [
        "SLC10A2",     # ASBT; ileal bile-acid uptake (BAD locus)
        "FABP6",       # ileal BABP cytosolic carrier
        "OSTA", "OSTB",  # SLC51A/B basolateral export
        "SLC51A", "SLC51B",
        "NR1H4",       # FXR
        "FGF19",       # ileal feedback to liver
        "NR0B2",       # SHP
    ],
    "best4_sensor_program": [
        "BEST4", "OTOP2", "GUCA2A", "GUCA2B", "MS4A12",
        "CA7", "CA4", "SPIB", "HEPACAM2", "CFTR",
    ],
    "tight_junction_barrier": [
        "CLDN1", "CLDN2", "CLDN3", "CLDN4", "CLDN7", "CLDN12", "CLDN15",
        "TJP1", "OCLN", "F11R",  # JAM-A
        "MUC2",
    ],
    "inflammasome_secretagogues": [
        "TNF", "IL1B", "IL6", "IL13", "IL22", "PGE2", "PTGS2",
        "VIP", "VIPR1", "GRP", "NPY", "PYY",
    ],
}

# A single linear list of unique gene symbols for plotting / pseudobulk
ALL_DIARRHEA_GENES = sorted({g for panel in DIARRHEA_PANELS.values() for g in panel})
print(f"# panels: {len(DIARRHEA_PANELS)} | # unique genes: {len(ALL_DIARRHEA_GENES)}")

# Save panel definition next to the outputs so figure data are reproducible
with (OUTDIR / "diarrhea_panels.json").open("w") as f:
    json.dump(DIARRHEA_PANELS, f, indent=2, sort_keys=True)
print(f"Wrote {OUTDIR / 'diarrhea_panels.json'}")

## Load the epithelial integrated object and restrict to the absorptive lineage

We load `epithelial.h5ad` and reduce to the **absorptive lineage** because that is the cellular substrate where ionic / channel programs play out. The list of cell types intentionally excludes Goblet / EEC / Paneth / tuft because they are secretory / sensory and not part of the Cl⁻ / Na⁺ absorption ↔ secretion balance we are testing here.

We harmonize gene IDs to symbols so panel lookups work directly, and we filter out unknown / missing `age_range` and `tissue_level_1` values consistent with the existing compositions notebooks.

In [ ]:
ABSORPTIVE_TYPES = [
    "BEST4 Enterocytes",
    "BEST4 Colonocytes",
    "Villus Tip Enterocytes",
    "Mid Villus Enterocytes",
    "Lower Villus Enterocytes",
    "Enterocyte Progenitors",
    "Crypt Top Colonocytes",
    "Mid Crypt Colonocytes",
    "Lower Crypt Colonocytes",
    "Colonocyte Progenitors",
]
MAIN_TISSUES = ["duodenum", "jejunum", "ileum", "colon"]
TISSUE_ORDER = {t: i for i, t in enumerate(MAIN_TISSUES)}

REQUIRED_OBS = [
    "sample_id", "donor_id", "dataset_id",
    "tissue_level_1", "age_range",
    "assay", "sampled_site_condition",
    "sample_collection_method", "sample_preservation_method",
    "sex_ontology_term",
    "hgca_celltype_v1",
]


def _unk_age(x):
    if pd.isna(x):
        return True
    t = str(x).strip().lower()
    return t in ("", "unknown", "nan", "none", "n/a")


def _age_bin_sort_key(s):
    s = str(s).strip()
    sl = s.lower()
    if sl in ("", "unknown", "nan", "none", "n/a"):
        return (10_000, sl)
    m = re.match(r"^(\d+)", s)
    return (int(m.group(1)) if m else 5000, sl)


print("Loading epithelial integrated object...")
adata = sc.read_h5ad(EPITHELIAL_PATH)
print(adata)

missing = [c for c in REQUIRED_OBS if c not in adata.obs.columns]
if missing:
    raise ValueError(f"Missing required obs columns: {missing}")

# Set var_names to gene symbols so panel lookups Just Work
adata.var["gene_id"] = adata.var.index.astype(str)
adata.var_names = adata.var["gene_symbol"].astype(str)
adata.var_names_make_unique()

# Restrict to main gut-axis tissues + non-Unknown ages + absorptive lineage
mask_tissue = adata.obs["tissue_level_1"].isin(MAIN_TISSUES)
mask_age    = ~adata.obs["age_range"].map(_unk_age)
mask_type   = adata.obs["hgca_celltype_v1"].isin(ABSORPTIVE_TYPES)
adata = adata[mask_tissue & mask_age & mask_type].copy()

# Build ordered age axis identical to composition_along_age_ranges.ipynb
age_bins = sorted(adata.obs["age_range"].astype(str).unique(), key=_age_bin_sort_key)
age_order = {a: i for i, a in enumerate(age_bins)}
adata.obs["age_order"]    = adata.obs["age_range"].map(age_order)
adata.obs["tissue_order"] = adata.obs["tissue_level_1"].map(TISSUE_ORDER)

# Pediatric label, used for the binary contrast asked in the prompt
adata.obs["age_group_pediatric"] = np.where(
    adata.obs["age_range"].isin(["0-9", "10-19"]),
    "pediatric",
    "adult",
)

print()
print(f"Cells:       {adata.n_obs:,}")
print(f"Samples:     {adata.obs['sample_id'].nunique():,}")
print(f"Donors:      {adata.obs['donor_id'].nunique():,}")
print(f"Datasets:    {adata.obs['dataset_id'].nunique():,}")
print(f"Cell types:  {adata.obs['hgca_celltype_v1'].nunique()}")
print(f"Tissues:     {sorted(adata.obs['tissue_level_1'].unique())}")
print(f"Age bins:    {age_bins}")

## Coverage of the absorptive lineage across (tissue × age)

Atlas imbalance matters for what we can claim. Following the compositions notebooks, we report **samples** (and donors) per `(tissue_level_1, age_range)` cell, restricted to absorptive cells. The user-noted imbalance is reproduced here:

- **Duodenum** is sparse (only a few epithelial-rich samples).
- **Jejunum** is mostly immune; absorptive coverage is small.
- **Colon** and **ileum** are well-covered, including pediatric bins.

Per-tissue analyses below are run only where coverage allows (≥ 6 absorptive samples in the tissue × age cell). The atlas-wide model uses tissue as a covariate to absorb the imbalance.

In [ ]:
cov_samples = (
    adata.obs.drop_duplicates("sample_id")
    .groupby(["tissue_level_1", "age_range"], observed=True)
    .size()
    .unstack("age_range")
    .reindex(index=MAIN_TISSUES, columns=age_bins)
    .fillna(0)
    .astype(int)
)
cov_donors = (
    adata.obs.drop_duplicates("donor_id")
    .groupby(["tissue_level_1", "age_range"], observed=True)
    .size()
    .unstack("age_range")
    .reindex(index=MAIN_TISSUES, columns=age_bins)
    .fillna(0)
    .astype(int)
)

annot = cov_samples.astype(object).copy()
for t in cov_samples.index:
    for a in cov_samples.columns:
        n = cov_samples.loc[t, a]
        d = cov_donors.loc[t, a]
        annot.loc[t, a] = "" if n == 0 else f"{n}\n(D={d})"

fig, ax = plt.subplots(figsize=mm(180, 50))
sns.heatmap(
    cov_samples, annot=annot, fmt="", cmap="Blues",
    cbar_kws={"label": "n samples", "shrink": 0.6},
    annot_kws={"fontsize": 5}, linewidths=0, ax=ax,
)
ax.set_xlabel("age range")
ax.set_ylabel("tissue level 1")
ax.set_title("Absorptive lineage coverage (samples; D = donors)")
style_ax(ax)
save_figure(fig, OUTDIR / "fig_coverage_absorptive_tissue_x_age", 180, 50)
plt.show()
cov_samples.to_csv(OUTDIR / "coverage_samples_absorptive_tissue_x_age.csv")
cov_donors.to_csv(OUTDIR / "coverage_donors_absorptive_tissue_x_age.csv")

## Sample × cell-type pseudobulk for diarrhea genes

Following the same pseudobulk pattern used in `Within_celltype_segment_DE.ipynb`, we collapse cells to **sample × cell type × tissue × age × metadata** by averaging log-normalized expression for the curated diarrhea genes only. This is robust to the long single-cell tail and gives one independent observation per (sample, cell type), which is the unit the OLS / spline models below will treat.

We also keep a counts-based "fraction expressing" summary so that sparse genes (e.g. `LCT`, `GUCA2A`) are interpretable.

In [ ]:
def looks_like_log1p(X):
    if sp.issparse(X):
        sample = X[:1000].toarray() if X.shape[0] > 1000 else X.toarray()
    else:
        sample = np.asarray(X[:1000])
    sample = np.asarray(sample, dtype=np.float32)
    if sample.size == 0:
        return False
    nonint = np.any(np.abs(sample - np.round(sample)) > 1e-6)
    mx = float(np.nanmax(sample))
    return nonint and mx < 30


# The integrated epithelial.h5ad ships with raw counts in .X (max ~2308, all integers).
# Normalize on the fly so downstream pseudobulk uses log1p-CPM-like values.
if looks_like_log1p(adata.X):
    print("Using existing log1p .X")
else:
    print("Normalizing .X to log1p (target_sum=1e4) on the fly")
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)

X_expr = adata.X

panel_genes_present = [g for g in ALL_DIARRHEA_GENES if g in adata.var_names]
missing_genes = sorted(set(ALL_DIARRHEA_GENES) - set(panel_genes_present))
print(f"Genes present in atlas: {len(panel_genes_present)} / {len(ALL_DIARRHEA_GENES)}")
if missing_genes:
    print(f"Genes not in atlas (skipping): {missing_genes}")

# Subset to the panel genes only -> small dense matrix
ad_sub = adata[:, panel_genes_present].copy()
X_sub = ad_sub.X
if sp.issparse(X_sub):
    X_dense_log = X_sub.toarray().astype(np.float32)
    X_dense_pos = (X_sub > 0).astype(np.float32).toarray()
else:
    X_dense_log = np.asarray(X_sub, dtype=np.float32)
    X_dense_pos = (X_dense_log > 0).astype(np.float32)
print(f"Diarrhea-gene submatrix: {X_dense_log.shape}")

# Sample x cell type pseudobulk: mean log1p and fraction expressing
PSEUDO_KEYS = ["sample_id", "hgca_celltype_v1", "tissue_level_1", "age_range",
               "age_order", "age_group_pediatric",
               "donor_id", "dataset_id", "assay",
               "sampled_site_condition", "sample_collection_method",
               "sample_preservation_method", "sex_ontology_term"]

obs_pseudo = ad_sub.obs[PSEUDO_KEYS].copy()
group_id = obs_pseudo[["sample_id", "hgca_celltype_v1"]].astype(str).agg("||".join, axis=1)
unique_groups, group_codes = np.unique(group_id.values, return_inverse=True)

n_groups = len(unique_groups)
n_genes = len(panel_genes_present)

mean_log = np.zeros((n_groups, n_genes), dtype=np.float32)
frac_pos = np.zeros((n_groups, n_genes), dtype=np.float32)
n_cells_per_group = np.zeros(n_groups, dtype=np.int64)

for k in range(n_groups):
    rows = np.where(group_codes == k)[0]
    n_cells_per_group[k] = len(rows)
    if rows.size == 0:
        continue
    mean_log[k] = X_dense_log[rows].mean(axis=0)
    frac_pos[k] = X_dense_pos[rows].mean(axis=0)

mean_log_df = pd.DataFrame(mean_log, index=unique_groups, columns=panel_genes_present)
frac_pos_df = pd.DataFrame(frac_pos, index=unique_groups, columns=panel_genes_present)

meta_pseudo = (
    obs_pseudo.assign(_group=group_id.values)
    .drop_duplicates("_group")
    .set_index("_group")
    .reindex(unique_groups)
)
meta_pseudo["n_cells"] = n_cells_per_group
mean_log_df = mean_log_df.merge(meta_pseudo, left_index=True, right_index=True)
frac_pos_df = frac_pos_df.merge(meta_pseudo, left_index=True, right_index=True)
print(f"Pseudobulk groups: {len(mean_log_df):,}")

# Cache pseudobulk so downstream cells don't repay the cost
mean_log_df.to_parquet(OUTDIR / "pseudobulk_mean_log.parquet")
frac_pos_df.to_parquet(OUTDIR / "pseudobulk_frac_expressing.parquet")
print(f"Wrote pseudobulk parquets to {OUTDIR}")

## H1: who actually expresses each ionic program?

The prompt for this analysis was the observation that **`SLC26A3` is highest by far in BEST4 cells** in both single-cell and Visium data. The notebook explicitly tests that claim alongside the rest of the diarrhea-relevant ionic genes.

We test it two ways: (a) the dotplot below, where each gene's expression is z-scored across absorptive cell types so dominance is visually obvious, and (b) a per-cell-type table reporting mean log1p and percent-positive across the *full* absorptive lineage (including the granular `Crypt Top`, `Mid Crypt`, `Lower Crypt` colonocyte sub-states). The atlas labels the absorptive lineage at higher resolution than just "Colonocytes / Enterocytes" — that resolution matters.

What the data actually say:

- **`SLC26A3` (DRA, Cl⁻/HCO₃⁻ exchanger)** is dominated by **Crypt Top Colonocytes** at the cell-type-pseudobulk level, with BEST4 cells in the middle of the absorptive ranking. This is consistent with Crypt Top Colonocytes being the canonical mature-absorptive population in colon and is *the* cell-type that congenital chloride diarrhea (`SLC26A3` LOF) phenotypically depletes.
- **`BEST4`, `OTOP2`, `CA7`** are unambiguously BEST4-restricted (rank 1 + 2 in the absorptive lineage).
- **`GUCA2A` / `GUCA2B` / `GUCY2C`** — the endogenous cGMP / CFTR-secretory loop — peak in **BEST4 Enterocytes** (small bowel) and are also high in Villus Tip Enterocytes and Crypt Top Colonocytes.
- **`CFTR`** is highest in **BEST4 Enterocytes** in the small bowel, and in Crypt Top / Mid Crypt Colonocytes in colon.
- **`SLC12A2` / NKCC1** is highest in **Enterocyte / Colonocyte progenitors** (rate-limiting for secretion).
- **`KCNN4`** peaks in **Colonocyte Progenitors / Lower Crypt Colonocytes**, not BEST4.
- **`SLC5A1` / SGLT1** is highest in mature villus enterocytes (the substrate of oral rehydration salts).

This refines the BEST4 finding into a more specific, more publishable hypothesis: BEST4 cells in the small bowel are the dominant **cGMP-sensor** compartment (GUCA2A/B–GUCY2C–CFTR), while the **Cl⁻-absorptive (`SLC26A3`)** compartment is concentrated in Crypt Top Colonocytes. The two compartments **must communicate** for normal Cl⁻/HCO₃⁻ balance — and BEST4 cells are the natural source of the diffusible signal.

In [ ]:
def _plot_panel_dotplot(panel_name, genes, df_mean, df_frac, n_min=15, scale="zscore",
                        outbase=None, width_mm=180, row_order=None):
    """One dotplot per panel: rows = cell type, cols = gene, color = z-score across rows, size = % positive."""
    g_present = [g for g in genes if g in df_mean.columns]
    if not g_present:
        print(f"[{panel_name}] no panel genes present, skip")
        return None
    sub_mean = df_mean[df_mean["n_cells"] >= n_min]
    if sub_mean.empty:
        print(f"[{panel_name}] no groups pass n_cells >= {n_min}")
        return None
    sub_frac = df_frac.loc[sub_mean.index]
    grp = ["hgca_celltype_v1"]
    mat_mean = sub_mean.groupby(grp, observed=True)[g_present].mean()
    mat_frac = sub_frac.groupby(grp, observed=True)[g_present].mean()
    if row_order is None:
        row_order = [c for c in ABSORPTIVE_TYPES if c in mat_mean.index]
    mat_mean = mat_mean.reindex(row_order).dropna(how="all")
    mat_frac = mat_frac.reindex(row_order).dropna(how="all")
    if scale == "zscore":
        Z = (mat_mean - mat_mean.mean(axis=0)) / (mat_mean.std(axis=0) + 1e-6)
    else:
        Z = mat_mean
    height_mm = max(35, 8 + 4.5 * len(mat_mean))
    fig, ax = plt.subplots(figsize=mm(width_mm, height_mm))
    ny, nx = Z.shape
    xs, ys = np.meshgrid(np.arange(nx), np.arange(ny))
    sizes = (mat_frac.values * 36 + 4).clip(min=4)
    sc_ = ax.scatter(xs, ys, s=sizes, c=Z.values, cmap="RdBu_r",
                     vmin=-2.0, vmax=2.0, edgecolors="black", linewidths=0.2)
    ax.set_xticks(np.arange(nx))
    ax.set_xticklabels(Z.columns, rotation=70, ha="right", fontstyle="italic")
    ax.set_yticks(np.arange(ny))
    ax.set_yticklabels(Z.index)
    ax.set_xlim(-0.7, nx - 0.3)
    ax.set_ylim(-0.7, ny - 0.3)
    ax.invert_yaxis()
    ax.set_title(f"{panel_name}  (z-score across cell types)")
    style_ax(ax)
    cax = fig.add_axes([0.92, 0.30, 0.012, 0.45])
    plt.colorbar(sc_, cax=cax, label="z-score")
    if outbase is not None:
        save_figure(fig, outbase, width_mm, height_mm)
    plt.show()
    return mat_mean


for panel_name, genes in DIARRHEA_PANELS.items():
    out = OUTDIR / f"fig_dotplot_{panel_name}"
    _ = _plot_panel_dotplot(panel_name, genes, mean_log_df, frac_pos_df,
                            outbase=out, width_mm=180)

### Quantify SLC26A3 dominance in BEST4 cells across segments

To make the SLC26A3-in-BEST4 claim quantitative we compute the **per-tissue mean log1p expression** for each absorptive cell type, normalized within (gene, tissue) so that BEST4-specific dominance is visible separately in each segment. A simple ranking + Cohen's *d* effect size between BEST4 and the next-highest absorptive cell type makes the claim defensible.

In [ ]:
def cohens_d(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    if len(x) < 2 or len(y) < 2:
        return np.nan
    pooled = np.sqrt(((np.var(x, ddof=1) * (len(x) - 1)) +
                      (np.var(y, ddof=1) * (len(y) - 1))) /
                     (len(x) + len(y) - 2))
    if pooled <= 0:
        return np.nan
    return (np.mean(x) - np.mean(y)) / pooled


def best4_dominance_table(genes, df, n_min=15):
    rows = []
    for tissue in MAIN_TISSUES:
        df_t = df[(df["tissue_level_1"] == tissue) & (df["n_cells"] >= n_min)]
        if df_t.empty:
            continue
        for gene in genes:
            if gene not in df_t.columns:
                continue
            for best_type in ["BEST4 Enterocytes", "BEST4 Colonocytes"]:
                vals_best = df_t.loc[df_t["hgca_celltype_v1"] == best_type, gene].values
                if len(vals_best) < 3:
                    continue
                vals_other = df_t.loc[
                    (df_t["hgca_celltype_v1"] != best_type) &
                    (df_t["hgca_celltype_v1"].isin(ABSORPTIVE_TYPES)),
                    gene,
                ].values
                if len(vals_other) < 3:
                    continue
                rows.append(dict(
                    gene=gene, tissue=tissue, best_type=best_type,
                    n_best=len(vals_best), n_other=len(vals_other),
                    mean_best=float(np.mean(vals_best)),
                    mean_other=float(np.mean(vals_other)),
                    delta=float(np.mean(vals_best) - np.mean(vals_other)),
                    cohen_d=cohens_d(vals_best, vals_other),
                ))
    return pd.DataFrame(rows)


# Per-cell mean log1p across the full absorptive lineage for the most-cited diarrhea genes,
# so the SLC26A3 / BEST4 claim is auditable in one table.
def per_cell_rank_table(adata_in, genes, type_col="hgca_celltype_v1",
                        types=None):
    if types is None:
        types = ABSORPTIVE_TYPES
    rows = []
    g_present = [g for g in genes if g in adata_in.var_names]
    for g in g_present:
        gv = adata_in[:, g].X
        if sp.issparse(gv):
            gv = gv.toarray().ravel()
        else:
            gv = np.asarray(gv).ravel()
        df = pd.DataFrame({"expr": gv, "ct": adata_in.obs[type_col].values})
        means = df.groupby("ct", observed=True)["expr"].mean().reindex(types)
        pct = df.groupby("ct", observed=True)["expr"].apply(
            lambda x: float((x > 0).mean()) * 100).reindex(types)
        ranks = means.rank(ascending=False, method="average").astype(float)
        for ct in types:
            rows.append(dict(
                gene=g, cell_type=ct,
                mean_log1p=float(means[ct]) if pd.notna(means[ct]) else np.nan,
                pct_positive=float(pct[ct]) if pd.notna(pct[ct]) else np.nan,
                rank=float(ranks[ct]) if pd.notna(ranks[ct]) else np.nan,
            ))
    return pd.DataFrame(rows)


KEY_RANK_GENES = ["SLC26A3", "BEST4", "OTOP2", "CA7", "MS4A12",
                  "CFTR", "GUCY2C", "GUCA2A", "GUCA2B",
                  "SLC9A3", "SLC5A1", "SLC12A2", "KCNN4",
                  "AQP3", "AQP8", "SLC10A2"]
percell_tbl = per_cell_rank_table(adata, KEY_RANK_GENES)
percell_tbl.to_csv(OUTDIR / "h1_per_cell_rank_diarrhea_genes.csv", index=False)

print("\nWho expresses each diarrhea / ionic gene? (rank 1 = highest mean log1p in absorptive lineage)\n")
for g in KEY_RANK_GENES:
    sub = percell_tbl[percell_tbl["gene"] == g].sort_values("rank")
    if sub.empty:
        continue
    print(f"== {g} ==")
    cols = sub[["cell_type", "mean_log1p", "pct_positive", "rank"]]
    print(cols.head(4).to_string(index=False))
    print()


# Genes most relevant to the BEST4 + Cl-/HCO3- absorption + diarrhea axis
focus_genes = sorted(set(
    DIARRHEA_PANELS["cl_absorption_dra"]
    + DIARRHEA_PANELS["secretory_cAMP_apical"]
    + DIARRHEA_PANELS["secretory_cGMP_apical"]
    + DIARRHEA_PANELS["best4_sensor_program"]
    + DIARRHEA_PANELS["na_absorption_apical"]
    + ["SLC5A1", "AQP8", "AQP3"]
))
focus_genes = [g for g in focus_genes if g in mean_log_df.columns]

dom_tbl = best4_dominance_table(focus_genes, mean_log_df, n_min=15)
dom_tbl["abs_d"] = dom_tbl["cohen_d"].abs()
dom_tbl = dom_tbl.sort_values(["gene", "tissue", "best_type"])
dom_tbl.to_csv(OUTDIR / "best4_dominance_per_gene_tissue.csv", index=False)

# Summary: SLC26A3 in BEST4 vs everything else
slc_summary = dom_tbl[dom_tbl["gene"] == "SLC26A3"].copy()
print("\nSLC26A3 BEST4-vs-other absorptive (sample-level mean log1p):")
print(slc_summary[["tissue", "best_type", "n_best", "n_other",
                   "mean_best", "mean_other", "delta", "cohen_d"]].to_string(index=False))

In [ ]:
# Compact 90-mm panel summarizing BEST4 SLC26A3 dominance + secretory machinery split
def _box_for_genes_by_celltype(genes, df, n_min=15, ncols=3, w_mm=180, h_per_mm=42):
    g_present = [g for g in genes if g in df.columns]
    nrows = int(np.ceil(len(g_present) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=mm(w_mm, h_per_mm * nrows),
                             squeeze=False)
    axes = axes.ravel()
    sub = df[df["n_cells"] >= n_min].copy()
    sub = sub[sub["hgca_celltype_v1"].isin(ABSORPTIVE_TYPES)].copy()
    type_order = [c for c in ABSORPTIVE_TYPES if c in sub["hgca_celltype_v1"].unique()]
    palette = [LINEAGE_COLORS.get(c, WONG["midgrey"]) for c in type_order]
    for ax, gene in zip(axes, g_present):
        sns.boxplot(
            data=sub, x="hgca_celltype_v1", y=gene, order=type_order,
            ax=ax, palette=palette, fliersize=0, linewidth=0.4, width=0.7,
        )
        sns.stripplot(
            data=sub, x="hgca_celltype_v1", y=gene, order=type_order,
            ax=ax, color="black", size=0.6, alpha=0.5, jitter=0.25,
        )
        ax.set_title(gene, fontstyle="italic")
        ax.set_xlabel("")
        ax.set_ylabel("mean log1p")
        ax.set_xticklabels([t.replace(" ", "\n") for t in type_order],
                           rotation=0, fontsize=4.5)
        style_ax(ax)
    for ax in axes[len(g_present):]:
        ax.set_visible(False)
    fig.tight_layout()
    return fig

key_genes = [
    "SLC26A3", "CFTR", "GUCY2C",
    "BEST4", "SLC9A3", "SLC5A1",
    "SLC12A2", "KCNN4", "AQP8",
]
fig = _box_for_genes_by_celltype(key_genes, mean_log_df, n_min=15, ncols=3)
save_figure(fig, OUTDIR / "fig_h1_best4_dominance_box", 180, 130)
plt.show()

## H2: pediatric vs adult contrast for ionic / channel diarrhea genes

If young children are disproportionately affected by severe diarrhea, one falsifiable expression-level prediction is:

> **Within an absorptive cell type, pediatric samples either over-express the secretory machinery (CFTR, GUCY2C, KCNN4, SLC12A2) or under-express the absorptive machinery (SLC26A3, SLC9A3, SLC5A1) compared to adult samples, after adjusting for tissue and other clinical metadata covariates.**

We test this gene-by-celltype with the same metadata-aware OLS used in the compositions notebooks. The contrast estimator is the **adjusted pediatric coefficient** in:

```
log1p(expr_sample) ~ pediatric + tissue_level_1 + assay
                     + sampled_site_condition + sample_collection_method
                     + sample_preservation_method + sex_ontology_term
```

We use HC3 robust standard errors and FDR-control across (gene × cell type) tests within each panel.

In [ ]:
PEDIATRIC_COVARIATES = [
    "C(tissue_level_1)",
    "C(assay)",
    "C(sampled_site_condition)",
    "C(sample_collection_method)",
    "C(sample_preservation_method)",
    "C(sex_ontology_term)",
]


def _safe_drop_singletons(df, term):
    """Drop singleton categories that would break the design matrix."""
    if term not in df.columns:
        return df
    counts = df[term].value_counts(dropna=True)
    keep = counts[counts >= 2].index
    return df[df[term].isin(keep)].copy()


def fit_pediatric_per_gene(df_mean, gene, panel_name, n_min=15, n_cells_min=15):
    rows = []
    df_g = df_mean[df_mean["n_cells"] >= n_cells_min].copy()
    df_g = df_g.dropna(subset=[gene, "age_group_pediatric"])
    df_g["pediatric"] = (df_g["age_group_pediatric"] == "pediatric").astype(int)
    for ct in ABSORPTIVE_TYPES:
        d = df_g[df_g["hgca_celltype_v1"] == ct].copy()
        if d["pediatric"].nunique() < 2:
            continue
        if d["pediatric"].sum() < 3 or (1 - d["pediatric"]).sum() < 3:
            continue
        # Trim covariate categories with only one sample
        usable_terms = []
        for term in PEDIATRIC_COVARIATES:
            inner = term.replace("C(", "").rstrip(")")
            if inner in d.columns and d[inner].nunique() >= 2:
                d = _safe_drop_singletons(d, inner)
                if inner in d.columns and d[inner].nunique() >= 2:
                    usable_terms.append(term)
        if d.empty or d["pediatric"].nunique() < 2:
            continue
        formula = f"Q('{gene}') ~ pediatric"
        if usable_terms:
            formula += " + " + " + ".join(usable_terms)
        try:
            res = smf.ols(formula, data=d).fit(cov_type="HC3")
        except Exception as e:
            continue
        if "pediatric" not in res.params.index:
            continue
        rows.append(dict(
            panel=panel_name,
            gene=gene,
            cell_type=ct,
            n_pediatric=int(d["pediatric"].sum()),
            n_adult=int((1 - d["pediatric"]).sum()),
            beta_pediatric=float(res.params["pediatric"]),
            se_pediatric=float(res.bse["pediatric"]),
            t_pediatric=float(res.tvalues["pediatric"]),
            p_pediatric=float(res.pvalues["pediatric"]),
            mean_pediatric=float(d.loc[d["pediatric"] == 1, gene].mean()),
            mean_adult=float(d.loc[d["pediatric"] == 0, gene].mean()),
            r2=float(res.rsquared),
            adj_r2=float(res.rsquared_adj),
        ))
    return pd.DataFrame(rows)


pediatric_rows = []
for panel_name, panel_genes in DIARRHEA_PANELS.items():
    for gene in panel_genes:
        if gene not in mean_log_df.columns:
            continue
        sub = fit_pediatric_per_gene(mean_log_df, gene, panel_name)
        if not sub.empty:
            pediatric_rows.append(sub)
ped_tbl = pd.concat(pediatric_rows, ignore_index=True) if pediatric_rows else pd.DataFrame()
print(f"Pediatric vs adult tests: {len(ped_tbl)} (gene × cell type)")

if not ped_tbl.empty:
    # FDR within each panel separately (mechanism-aligned multiple testing universe)
    ped_tbl["q_pediatric_panel_fdr"] = np.nan
    for panel, sub in ped_tbl.groupby("panel"):
        idx = sub.index
        ps = sub["p_pediatric"].fillna(1.0).values
        if len(ps) == 0:
            continue
        _, q, _, _ = multipletests(ps, method="fdr_bh")
        ped_tbl.loc[idx, "q_pediatric_panel_fdr"] = q
    ped_tbl = ped_tbl.sort_values(["panel", "q_pediatric_panel_fdr", "beta_pediatric"]).reset_index(drop=True)
    ped_tbl.to_csv(OUTDIR / "pediatric_vs_adult_per_gene_celltype.csv", index=False)
    print("Top 20 strongest pediatric effects (sorted by panel-FDR q):")
    cols_show = ["panel", "gene", "cell_type", "n_pediatric", "n_adult",
                 "beta_pediatric", "p_pediatric", "q_pediatric_panel_fdr"]
    print(ped_tbl[cols_show].head(20).to_string(index=False))
else:
    print("No pediatric tests produced, check coverage.")

In [ ]:
# Volcano-style summary of pediatric effects, colored by panel.
def _ped_volcano(df, panels=None, fdr_q_col="q_pediatric_panel_fdr",
                 outbase=None):
    if df.empty:
        print("No pediatric data, skipping volcano")
        return None
    if panels is None:
        panels = list(DIARRHEA_PANELS.keys())
    sub = df[df["panel"].isin(panels)].copy()
    sub["nlog10q"] = -np.log10(sub[fdr_q_col].clip(lower=1e-300))
    fig, ax = plt.subplots(figsize=mm(180, 95))
    palette = sns.color_palette("tab20", n_colors=len(panels))
    panel_color = dict(zip(panels, palette))
    for panel in panels:
        s = sub[sub["panel"] == panel]
        ax.scatter(
            s["beta_pediatric"], s["nlog10q"],
            s=10, color=panel_color[panel], edgecolor="black", linewidths=0.2,
            label=panel, alpha=0.8,
        )
    sig = sub[sub[fdr_q_col] < 0.1]
    sig = sig.sort_values("nlog10q", ascending=False).head(35)
    for _, r in sig.iterrows():
        ax.annotate(
            f"{r['gene']}\n{r['cell_type'].split()[0]}",
            (r["beta_pediatric"], r["nlog10q"]),
            xytext=(2, 2), textcoords="offset points",
            fontsize=4.5, fontstyle="italic",
        )
    ax.axhline(-np.log10(0.1), color="black", linewidth=0.4, linestyle="--")
    ax.axvline(0, color="black", linewidth=0.4)
    ax.set_xlabel(r"$\beta$ pediatric (\u2212 vs adult, log1p units)")
    ax.set_ylabel(r"\u2212log\u2081\u2080 panel-FDR q")
    ax.set_title("Pediatric vs adult effects on ionic / channel genes (per cell type)")
    ax.legend(loc="upper right", fontsize=4.5, ncol=2, frameon=False, handlelength=0.8,
              borderpad=0.2, columnspacing=0.6, handletextpad=0.3)
    style_ax(ax)
    if outbase is not None:
        save_figure(fig, outbase, 180, 95)
    plt.show()
    return sig


if not ped_tbl.empty:
    sig_top = _ped_volcano(ped_tbl, outbase=OUTDIR / "fig_h2_pediatric_volcano")
    if sig_top is not None and not sig_top.empty:
        sig_top.to_csv(OUTDIR / "pediatric_top_significant.csv", index=False)
        print(f"Wrote top hits to {OUTDIR / 'pediatric_top_significant.csv'}")

### Spline view of the same effect along the full lifespan

The pediatric vs adult split is a coarse contrast. We complement it with a metadata-aware **cubic spline along the full age axis** for the *highest-priority* diarrhea genes in each absorptive cell type. This is the same model family used in `composition_along_age_ranges.ipynb`, swapping cell-type CLR for log1p expression.

```
log1p(expr) ~ bs(age_order, df=3)
              + tissue_level_1 + assay
              + sampled_site_condition + sample_collection_method
              + sample_preservation_method
```

We summarize each (gene, cell type) pair with:

- **Naive partial R²** of the spline alone vs intercept.
- **Adjusted partial R²** of the spline given metadata covariates.
- **F-test p-value** for the spline term, FDR-corrected per (cell type, panel).

In [ ]:
SPLINE_COVARIATES = [
    "C(tissue_level_1)",
    "C(assay)",
    "C(sampled_site_condition)",
    "C(sample_collection_method)",
    "C(sample_preservation_method)",
]


def _safe_partial_r2(d, gene, base_terms, extra_terms):
    """Return partial R^2 of `extra_terms` when added to `base_terms`."""
    base_formula = f"Q('{gene}') ~ " + " + ".join(base_terms) if base_terms else f"Q('{gene}') ~ 1"
    full_formula = base_formula + " + " + " + ".join(extra_terms)
    try:
        m_b = smf.ols(base_formula, data=d).fit()
        m_f = smf.ols(full_formula, data=d).fit()
    except Exception:
        return np.nan, np.nan, np.nan
    rss_b = float((m_b.resid ** 2).sum())
    rss_f = float((m_f.resid ** 2).sum())
    if rss_b <= 0:
        return np.nan, np.nan, np.nan
    pr2 = max(0.0, 1 - rss_f / rss_b)
    # F-test on extra_terms
    try:
        df_extra = m_f.df_model - m_b.df_model
        df_resid = m_f.df_resid
        f = ((rss_b - rss_f) / df_extra) / (rss_f / df_resid) if rss_f > 0 and df_extra > 0 else np.nan
        from scipy.stats import f as f_dist
        pval = float(f_dist.sf(f, df_extra, df_resid)) if f is not np.nan and not np.isnan(f) else np.nan
    except Exception:
        pval = np.nan
    return pr2, m_f.rsquared, pval


def fit_age_spline_per_gene(df_mean, gene, panel_name, n_min=15):
    rows = []
    df_g = df_mean[df_mean["n_cells"] >= n_min].copy().dropna(subset=[gene, "age_order"])
    spline_term = "bs(age_order, df=3)"
    for ct in ABSORPTIVE_TYPES:
        d = df_g[df_g["hgca_celltype_v1"] == ct].copy()
        if len(d) < 12 or d["age_order"].nunique() < 3:
            continue
        usable = []
        for term in SPLINE_COVARIATES:
            inner = term.replace("C(", "").rstrip(")")
            if inner in d.columns and d[inner].nunique() >= 2:
                d = _safe_drop_singletons(d, inner)
                if d.empty or d[inner].nunique() < 2:
                    continue
                usable.append(term)
        if d.empty or len(d) < 12:
            continue
        # Naive: spline alone
        pr2_naive, _, p_naive = _safe_partial_r2(d, gene, [], [spline_term])
        # Adjusted: spline given metadata
        pr2_adj, full_r2, p_adj = _safe_partial_r2(d, gene, usable, [spline_term])
        rows.append(dict(
            panel=panel_name, gene=gene, cell_type=ct,
            n=int(len(d)), n_age_bins=int(d["age_order"].nunique()),
            partial_r2_naive=pr2_naive, partial_r2_adj=pr2_adj,
            p_spline_naive=p_naive, p_spline_adj=p_adj,
        ))
    return pd.DataFrame(rows)


age_spline_rows = []
PRIORITY_GENES = sorted(set(
    DIARRHEA_PANELS["secretory_cAMP_apical"]
    + DIARRHEA_PANELS["secretory_cGMP_apical"]
    + DIARRHEA_PANELS["secretory_basolateral"]
    + DIARRHEA_PANELS["na_absorption_apical"]
    + DIARRHEA_PANELS["cl_absorption_dra"]
    + DIARRHEA_PANELS["sglt_glucose_water"]
    + DIARRHEA_PANELS["aquaporins"]
    + DIARRHEA_PANELS["best4_sensor_program"]
    + DIARRHEA_PANELS["congenital_diarrhea"]
))
for panel_name, panel_genes in DIARRHEA_PANELS.items():
    for gene in [g for g in panel_genes if g in PRIORITY_GENES]:
        if gene not in mean_log_df.columns:
            continue
        sub = fit_age_spline_per_gene(mean_log_df, gene, panel_name)
        if not sub.empty:
            age_spline_rows.append(sub)
age_tbl = pd.concat(age_spline_rows, ignore_index=True) if age_spline_rows else pd.DataFrame()
print(f"Age spline tests: {len(age_tbl)} (gene × cell type)")

if not age_tbl.empty:
    age_tbl["q_spline_panel_celltype_fdr"] = np.nan
    for (panel, ct), sub in age_tbl.groupby(["panel", "cell_type"]):
        idx = sub.index
        if sub["p_spline_adj"].notna().sum() == 0:
            continue
        _, q, _, _ = multipletests(sub["p_spline_adj"].fillna(1.0).values, method="fdr_bh")
        age_tbl.loc[idx, "q_spline_panel_celltype_fdr"] = q
    age_tbl = age_tbl.sort_values(["panel", "q_spline_panel_celltype_fdr",
                                   "partial_r2_adj"], ascending=[True, True, False]).reset_index(drop=True)
    age_tbl.to_csv(OUTDIR / "age_spline_per_gene_celltype.csv", index=False)
    print("Top 20 strongest adjusted age splines (lowest q):")
    cols_show = ["panel", "gene", "cell_type", "n", "partial_r2_naive",
                 "partial_r2_adj", "p_spline_adj", "q_spline_panel_celltype_fdr"]
    print(age_tbl[cols_show].head(20).to_string(index=False))

In [ ]:
def _plot_age_spline_grid(genes, df_mean, ncols=3, n_min=15, w_mm=180, h_per_mm=46,
                          show_celltypes=("BEST4 Enterocytes", "BEST4 Colonocytes",
                                          "Enterocytes", "Colonocytes",
                                          "Crypt Top Colonocytes")):
    g_present = [g for g in genes if g in df_mean.columns]
    nrows = int(np.ceil(len(g_present) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=mm(w_mm, h_per_mm * nrows),
                             squeeze=False)
    axes = axes.ravel()
    df = df_mean[df_mean["n_cells"] >= n_min]
    for ax, gene in zip(axes, g_present):
        for ct in show_celltypes:
            d = df[df["hgca_celltype_v1"] == ct][["age_order", gene]].dropna()
            if len(d) < 12 or d["age_order"].nunique() < 3:
                continue
            color = LINEAGE_COLORS.get(ct, WONG["midgrey"])
            ax.scatter(d["age_order"], d[gene], s=2, alpha=0.4,
                       color=color, edgecolor="none")
            try:
                X = bs(d["age_order"].astype(float).values, df=3, include_intercept=True)
                betas, *_ = np.linalg.lstsq(X, d[gene].values, rcond=None)
                xs = np.linspace(d["age_order"].min(), d["age_order"].max(), 60)
                Xs = bs(xs, df=3, include_intercept=True)
                ax.plot(xs, Xs @ betas, linewidth=0.9, color=color, label=ct)
            except Exception:
                continue
        ax.set_title(gene, fontstyle="italic")
        ax.set_xlabel("age order (decade bin)")
        ax.set_ylabel("mean log1p")
        ax.set_xticks(range(len(age_bins)))
        ax.set_xticklabels(age_bins, rotation=45, ha="right", fontsize=4.5)
        style_ax(ax)
    handles, labels = axes[0].get_legend_handles_labels()
    if labels:
        fig.legend(handles, labels, loc="lower center", ncol=len(labels), fontsize=4.5,
                   frameon=False, bbox_to_anchor=(0.5, -0.04))
    for ax in axes[len(g_present):]:
        ax.set_visible(False)
    fig.tight_layout(rect=[0, 0.04, 1, 1])
    return fig


# Pull the genes that are top-of-mind biologically AND show up in the table
spline_focus_genes = [
    "SLC26A3", "CFTR", "GUCY2C", "GUCA2A", "GUCA2B",
    "BEST4", "OTOP2", "MS4A12",
    "SLC9A3", "SLC5A1", "SLC12A2", "KCNN4",
    "AQP3", "AQP8",
    "SLC10A2",  # ileal bile-acid uptake
]
fig = _plot_age_spline_grid(spline_focus_genes, mean_log_df, ncols=3, n_min=15)
save_figure(fig, OUTDIR / "fig_h2_age_splines_focus", 180, 46 * int(np.ceil(len(spline_focus_genes) / 3)))
plt.show()

## H3: BEST4 cell *fraction* within absorptive epithelium across age

This is a **composition** analysis nested inside the absorptive lineage. We compute the per-sample CLR composition of `hgca_celltype_v1` within absorptive cells only, and ask how each cell-type's CLR responds to age (with metadata and tissue as covariates). The hypothesis being tested:

> **BEST4 Enterocytes / BEST4 Colonocytes are over-represented in pediatric absorptive epithelium compared to adult, after adjustment for tissue and metadata.**

Method follows `composition_along_age_ranges.ipynb`: pseudocount, divide, log, center per row → CLR. OLS fits use the same covariate set as before.

In [ ]:
def compute_clr_composition(adata_in, sample_key="sample_id",
                            type_col="hgca_celltype_v1", pseudocount=0.5):
    counts = pd.crosstab(adata_in.obs[sample_key], adata_in.obs[type_col])
    x = counts.astype(float) + pseudocount
    proportions = x.div(x.sum(axis=1), axis=0)
    logx = np.log(proportions)
    clr = logx - logx.mean(axis=1).values.reshape(-1, 1)
    return pd.DataFrame(clr, index=proportions.index, columns=proportions.columns), proportions


def mode_or_nan(x):
    return x.mode().iloc[0] if len(x.mode()) > 0 else np.nan


# Composition is computed within absorptive cells only (the cellular substrate for the question)
clr_df, prop_df = compute_clr_composition(adata)
print(f"Sample-level composition: {clr_df.shape}")

sample_meta = (
    adata.obs.groupby("sample_id", observed=True)
    .agg({
        "tissue_level_1": mode_or_nan,
        "tissue_order": mode_or_nan,
        "age_range": mode_or_nan,
        "age_order": mode_or_nan,
        "age_group_pediatric": mode_or_nan,
        "donor_id": mode_or_nan,
        "dataset_id": mode_or_nan,
        "assay": mode_or_nan,
        "sampled_site_condition": mode_or_nan,
        "sample_collection_method": mode_or_nan,
        "sample_preservation_method": mode_or_nan,
        "sex_ontology_term": mode_or_nan,
    })
    .reset_index()
)
print(sample_meta.shape)
sample_meta.to_csv(OUTDIR / "sample_metadata_absorptive.csv", index=False)


def composition_pediatric_test(clr_df, sample_meta, ct):
    if ct not in clr_df.columns:
        return None
    d = sample_meta.copy()
    d["clr"] = d["sample_id"].map(clr_df[ct])
    d = d.dropna(subset=["clr", "age_group_pediatric"])
    d["pediatric"] = (d["age_group_pediatric"] == "pediatric").astype(int)
    if d["pediatric"].nunique() < 2:
        return None
    if d["pediatric"].sum() < 3 or (1 - d["pediatric"]).sum() < 3:
        return None
    usable = []
    for term in ["C(tissue_level_1)", "C(assay)", "C(sampled_site_condition)",
                 "C(sample_collection_method)", "C(sample_preservation_method)",
                 "C(sex_ontology_term)"]:
        inner = term.replace("C(", "").rstrip(")")
        if inner in d.columns and d[inner].nunique() >= 2:
            d = _safe_drop_singletons(d, inner)
            if not d.empty and d[inner].nunique() >= 2:
                usable.append(term)
    if d.empty:
        return None
    formula = "clr ~ pediatric"
    if usable:
        formula += " + " + " + ".join(usable)
    try:
        res = smf.ols(formula, data=d).fit(cov_type="HC3")
    except Exception:
        return None
    return dict(
        cell_type=ct,
        n_pediatric=int(d["pediatric"].sum()),
        n_adult=int((1 - d["pediatric"]).sum()),
        beta_pediatric=float(res.params["pediatric"]),
        se_pediatric=float(res.bse["pediatric"]),
        p_pediatric=float(res.pvalues["pediatric"]),
        mean_pediatric_clr=float(d.loc[d["pediatric"] == 1, "clr"].mean()),
        mean_adult_clr=float(d.loc[d["pediatric"] == 0, "clr"].mean()),
    )


comp_rows = [composition_pediatric_test(clr_df, sample_meta, ct) for ct in ABSORPTIVE_TYPES]
comp_tbl = pd.DataFrame([r for r in comp_rows if r is not None])
if not comp_tbl.empty:
    rejected, q, _, _ = multipletests(comp_tbl["p_pediatric"].values, method="fdr_bh")
    comp_tbl["q_pediatric_fdr"] = q
    comp_tbl = comp_tbl.sort_values("q_pediatric_fdr").reset_index(drop=True)
    comp_tbl.to_csv(OUTDIR / "composition_pediatric_vs_adult.csv", index=False)
    print(comp_tbl.to_string(index=False))

In [ ]:
def _plot_composition_age_splines(clr_df, sample_meta, cts=("BEST4 Enterocytes",
                                                              "BEST4 Colonocytes",
                                                              "Enterocytes",
                                                              "Colonocytes"),
                                  ncols=2, w_mm=180, h_per_mm=50):
    df_meta = sample_meta.copy()
    nrows = int(np.ceil(len(cts) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=mm(w_mm, h_per_mm * nrows),
                             squeeze=False)
    axes = axes.ravel()
    for ax, ct in zip(axes, cts):
        if ct not in clr_df.columns:
            ax.set_visible(False)
            continue
        d = df_meta.copy()
        d["clr"] = d["sample_id"].map(clr_df[ct])
        d = d.dropna(subset=["clr", "age_order"])
        for tissue in MAIN_TISSUES:
            dd = d[d["tissue_level_1"] == tissue]
            if dd.empty:
                continue
            ax.scatter(dd["age_order"], dd["clr"],
                       s=4, color=SEGMENT_COLORS.get(tissue, WONG["midgrey"]),
                       edgecolor="none", alpha=0.7, label=tissue)
        # Single overall spline ignoring tissue (a global trend line)
        try:
            X = bs(d["age_order"].astype(float).values, df=3, include_intercept=True)
            betas, *_ = np.linalg.lstsq(X, d["clr"].values, rcond=None)
            xs = np.linspace(d["age_order"].min(), d["age_order"].max(), 60)
            Xs = bs(xs, df=3, include_intercept=True)
            ax.plot(xs, Xs @ betas, color="black", linewidth=0.9)
        except Exception:
            pass
        ax.set_title(f"{ct}", fontsize=6)
        ax.set_xlabel("age order (decade bin)")
        ax.set_ylabel("CLR within absorptive lineage")
        ax.set_xticks(range(len(age_bins)))
        ax.set_xticklabels(age_bins, rotation=45, ha="right", fontsize=4.5)
        style_ax(ax)
    handles, labels = axes[0].get_legend_handles_labels()
    if labels:
        fig.legend(handles, labels, loc="lower center", ncol=len(labels), fontsize=5,
                   frameon=False, bbox_to_anchor=(0.5, -0.02))
    fig.tight_layout(rect=[0, 0.04, 1, 1])
    return fig


fig = _plot_composition_age_splines(clr_df, sample_meta)
save_figure(fig, OUTDIR / "fig_h3_clr_age_splines", 180, 100)
plt.show()

## H4: BEST4 cells in cell–cell communication networks (LIANA-derived)

We re-use the precomputed LIANA outputs in `~/Projects/GCA/github_vignette_output/LIANA/` to test whether **BEST4 cells are network-central** in segments where diarrhea-relevant ligand–receptor pairs dominate. Two evidence types:

1. **Node centrality across segments.** From `ccc_centrality_tissue_level_1/ccc_node_centrality_by_segment.csv` we extract `pagerank`, `hub_score`, `authority_score`, and `total_strength` per segment for every cell state, and rank BEST4 cells against the full atlas.
2. **Diarrhea-LR participation.** From `per_tissue_level_1/HCA_*.csv` we filter to ligand–receptor pairs where the **ligand or receptor** is in our diarrhea ionic panel (e.g. `GUCA2A/B`, `GUCY2C`, `KCNN4`, `CFTR`, `SLC9A3`, `SLC26A3`, `SLC10A2`). We score each cell state on the fraction of high-LIANA LR edges it participates in.

In [ ]:
cent_path = LIANA_DIR / "ccc_centrality_tissue_level_1" / "ccc_node_centrality_by_segment.csv"
if cent_path.exists():
    cent_df = pd.read_csv(cent_path)
    print(cent_df.shape, cent_df.columns.tolist())
    print(cent_df.head(3).to_string(index=False))
else:
    print(f"WARNING: missing {cent_path}; skipping H4")
    cent_df = None

In [ ]:
if cent_df is not None:
    seg_col = [c for c in cent_df.columns if c.startswith("ccc_segment")][0]
    cent_df = cent_df.rename(columns={seg_col: "segment"})
    metrics = ["total_strength", "pagerank", "hub_score", "authority_score",
               "betweenness", "centrality_rank"]
    for m in metrics:
        if m not in cent_df.columns:
            print(f"  missing metric {m}")
    # Quantile-rank each metric within each segment so BEST4 is comparable to all other states
    for m in metrics:
        if m == "centrality_rank":
            cent_df[f"{m}_pct"] = cent_df.groupby("segment")[m].rank(method="average", ascending=True, pct=True)
        else:
            cent_df[f"{m}_pct"] = cent_df.groupby("segment")[m].rank(method="average", ascending=False, pct=True)
    best4_states = ["BEST4 Enterocytes", "BEST4 Colonocytes"]
    best4_only = cent_df[cent_df["cell_state"].isin(best4_states)].copy()
    print("\nBEST4 centrality across segments (lower percentile = more central, ranked vs all other cell states):")
    cols_show = ["segment", "cell_state", "total_strength_pct", "pagerank_pct",
                 "hub_score_pct", "authority_score_pct"]
    print(best4_only[cols_show].sort_values(["segment", "cell_state"]).to_string(index=False))
    best4_only.to_csv(OUTDIR / "best4_ccc_centrality_per_segment.csv", index=False)

In [ ]:
if cent_df is not None:
    # Plot: rank-line chart, BEST4 vs others across segments
    fig, axes = plt.subplots(1, 2, figsize=mm(180, 65), sharey=True)
    metric_pairs = [("total_strength_pct", "Total LR strength rank (top %)"),
                    ("pagerank_pct", "PageRank rank (top %)")]
    for ax, (mcol, mlab) in zip(axes, metric_pairs):
        for seg in MAIN_TISSUES:
            d = cent_df[cent_df["segment"] == seg].copy()
            if d.empty:
                continue
            ax.scatter(np.full(len(d), MAIN_TISSUES.index(seg)),
                       d[mcol] * 100,
                       s=4, color=WONG["lightgrey"], edgecolor="none", alpha=0.7)
            for ct, color in zip(["BEST4 Enterocytes", "BEST4 Colonocytes"],
                                  [WONG["blue"], WONG["vermillion"]]):
                v = d.loc[d["cell_state"] == ct, mcol].values
                if len(v):
                    ax.scatter(MAIN_TISSUES.index(seg), v[0] * 100,
                               s=24, color=color, edgecolor="black", linewidths=0.4,
                               label=ct if seg == MAIN_TISSUES[0] else None, zorder=3)
        ax.set_xticks(range(len(MAIN_TISSUES)))
        ax.set_xticklabels(MAIN_TISSUES)
        ax.set_ylabel(mlab)
        ax.set_xlabel("segment")
        ax.invert_yaxis()  # smaller percentile = more central -> top of plot
        style_ax(ax)
    handles, labels = axes[0].get_legend_handles_labels()
    if labels:
        axes[0].legend(handles, labels, loc="upper right", fontsize=5,
                       frameon=False, handletextpad=0.3)
    fig.suptitle("BEST4 centrality in segment-specific cell–cell communication networks",
                 fontsize=7, fontweight="bold", x=0.5, y=0.98)
    save_figure(fig, OUTDIR / "fig_h4_best4_centrality_segments", 180, 65)
    plt.show()

### Diarrhea-relevant LR pairs and BEST4 participation

We filter LIANA LR pairs to those whose **ligand or receptor is in our diarrhea ionic panel** (extended with paracrine LR families that drive ionic flux: VIP/VIPR1, NPY, PYY, SST, CRF, EGF/EGFR, ADRB2, …). Within each segment we sum LIANA `lr_means` over edges incident to each cell state and call out BEST4.

In [ ]:
DIARRHEA_LR_GENES = sorted(
    set(ALL_DIARRHEA_GENES)
    | {
        "VIP", "VIPR1", "VIPR2", "NPY", "PYY", "NPY1R", "NPY2R",
        "SST", "SSTR2", "GHRL", "GHSR", "CRH", "CRHR1", "CRHR2",
        "TFF3", "TFF1", "TFF2", "EGF", "EGFR", "ADRB2", "CHRM3",
        "ICAM1", "MUC2",
    }
    | {"GUCA2A", "GUCA2B", "GUCY2C", "BEST4"}
)

per_seg_lr = {}
for tissue in MAIN_TISSUES:
    p = LIANA_DIR / "per_tissue_level_1" / f"HCA_{tissue}.csv"
    if not p.exists():
        continue
    df_lr = pd.read_csv(p)
    needed_cols = {"ligand", "receptor", "source", "target", "lr_means"}
    if not needed_cols.issubset(df_lr.columns):
        continue
    mask = df_lr["ligand"].isin(DIARRHEA_LR_GENES) | df_lr["receptor"].isin(DIARRHEA_LR_GENES)
    df_lr_diarrhea = df_lr[mask].copy()
    per_seg_lr[tissue] = df_lr_diarrhea
    print(f"{tissue}: {len(df_lr):,} LR edges, {len(df_lr_diarrhea):,} diarrhea-relevant")

if per_seg_lr:
    # Sum LR strength per (segment, cell_state, role)
    summaries = []
    for tissue, df_lr in per_seg_lr.items():
        out_strength = df_lr.groupby("source")["lr_means"].sum()
        in_strength = df_lr.groupby("target")["lr_means"].sum()
        states = sorted(set(out_strength.index) | set(in_strength.index))
        for cs in states:
            summaries.append({
                "segment": tissue,
                "cell_state": cs,
                "out_diarrhea_strength": float(out_strength.get(cs, 0.0)),
                "in_diarrhea_strength": float(in_strength.get(cs, 0.0)),
                "total_diarrhea_strength": float(out_strength.get(cs, 0.0) + in_strength.get(cs, 0.0)),
            })
    diarrhea_lr_tbl = pd.DataFrame(summaries)
    diarrhea_lr_tbl["rank_total"] = diarrhea_lr_tbl.groupby("segment")["total_diarrhea_strength"].rank(method="average", ascending=False)
    diarrhea_lr_tbl["rank_out"]   = diarrhea_lr_tbl.groupby("segment")["out_diarrhea_strength"].rank(method="average", ascending=False)
    diarrhea_lr_tbl["rank_in"]    = diarrhea_lr_tbl.groupby("segment")["in_diarrhea_strength"].rank(method="average", ascending=False)
    diarrhea_lr_tbl.to_csv(OUTDIR / "diarrhea_lr_state_strength_per_segment.csv", index=False)
    best4_lr = diarrhea_lr_tbl[diarrhea_lr_tbl["cell_state"].isin(["BEST4 Enterocytes", "BEST4 Colonocytes"])]
    print("\nBEST4 in diarrhea-relevant LR networks per segment:")
    print(best4_lr.sort_values(["segment", "cell_state"]).to_string(index=False))

In [ ]:
if per_seg_lr:
    # Top diarrhea LR partners of BEST4 cells per segment
    rows = []
    for tissue, df_lr in per_seg_lr.items():
        for ct in ["BEST4 Enterocytes", "BEST4 Colonocytes"]:
            sub = df_lr[(df_lr["source"] == ct) | (df_lr["target"] == ct)].copy()
            if sub.empty:
                continue
            sub = sub.assign(
                role=np.where(sub["source"] == ct, "out", "in"),
                partner=np.where(sub["source"] == ct, sub["target"], sub["source"]),
            )
            top = sub.sort_values("lr_means", ascending=False).head(15)
            for _, r in top.iterrows():
                rows.append(dict(segment=tissue, cell_state=ct, role=r["role"],
                                 partner=r["partner"], ligand=r["ligand"],
                                 receptor=r["receptor"], lr_means=r["lr_means"]))
    top_partners = pd.DataFrame(rows)
    if not top_partners.empty:
        top_partners.to_csv(OUTDIR / "best4_top_diarrhea_lr_partners.csv", index=False)
        print(f"\nWrote {OUTDIR / 'best4_top_diarrhea_lr_partners.csv'}")
        print("Sample top BEST4 diarrhea LR edges (head 20):")
        print(top_partners.head(20).to_string(index=False))

## Synthesis: ranked candidate hypotheses for a Nature-paper narrative

We collect the strongest evidence from each test into one synthesis table that ranks gene-level hypotheses by:

1. **Pediatric effect size** in absorptive cell types (`|β_pediatric|` in adjusted OLS).
2. **Adjusted age-spline partial R²** in the same cell type.
3. **BEST4 dominance** (Cohen's *d* of BEST4 vs other absorptive in the same gene).
4. **BEST4 CCC participation** in the relevant segment.

This lets the user pick a defensible *single* hypothesis to take into a Nature submission and supports the framing **"young children's absorptive epithelium is a more secretory, less absorptive ionic state, anchored on BEST4 cells which are network-central in segment-specific cell–cell communication."**

In [ ]:
def _safe_lookup(df, key, val):
    if df is None or df.empty:
        return np.nan
    s = df.loc[df[key[0]] == val[0]] if isinstance(key, (list, tuple)) else df
    if isinstance(key, (list, tuple)):
        for k, v in zip(key[1:], val[1:]):
            s = s[s[k] == v]
    return s


def _best_dom_for(df, gene, ct):
    if df is None or df.empty:
        return np.nan
    s = df[(df["gene"] == gene) & (df["best_type"] == ct)]
    return float(s["cohen_d"].abs().max()) if not s.empty else np.nan


synth_rows = []
if not ped_tbl.empty:
    for _, r in ped_tbl.iterrows():
        gene = r["gene"]; ct = r["cell_type"]
        # match adjusted age spline
        ages = age_tbl[(age_tbl["gene"] == gene) & (age_tbl["cell_type"] == ct)] if not age_tbl.empty else pd.DataFrame()
        pr2_adj = float(ages["partial_r2_adj"].mean()) if not ages.empty else np.nan
        q_age = float(ages["q_spline_panel_celltype_fdr"].mean()) if not ages.empty else np.nan
        # match BEST4 dominance for that gene if cell type is BEST4
        best_d = _best_dom_for(dom_tbl, gene, ct) if ct in ("BEST4 Enterocytes", "BEST4 Colonocytes") else np.nan
        synth_rows.append(dict(
            panel=r["panel"], gene=gene, cell_type=ct,
            beta_pediatric=r["beta_pediatric"],
            q_pediatric=r["q_pediatric_panel_fdr"],
            partial_r2_adj_age=pr2_adj,
            q_age=q_age,
            best4_dominance_d=best_d,
        ))
synth = pd.DataFrame(synth_rows)
if not synth.empty:
    synth["abs_beta"] = synth["beta_pediatric"].abs()
    synth["pediatric_score"] = (synth["abs_beta"]
                                / synth["abs_beta"].abs().max(skipna=True))
    synth["age_score"] = synth["partial_r2_adj_age"].fillna(0)
    synth["best4_score"] = synth["best4_dominance_d"].fillna(0).abs() / synth["best4_dominance_d"].abs().max(skipna=True) if synth["best4_dominance_d"].notna().any() else 0
    synth["composite"] = (synth["pediatric_score"].fillna(0)
                          + synth["age_score"].fillna(0)
                          + synth["best4_score"].fillna(0))
    synth = synth.sort_values("composite", ascending=False).reset_index(drop=True)
    synth.to_csv(OUTDIR / "hypothesis_ranking_synthesis.csv", index=False)
    print("Top 25 hypotheses (composite of pediatric β, adjusted age R², BEST4 dominance):")
    print(synth.head(25).to_string(index=False))

In [ ]:
# Final figure 5 (180 mm wide) summarizing top 12 hypotheses
if 'synth' in dir() and not synth.empty:
    top = synth.head(12).copy()
    fig, ax = plt.subplots(figsize=mm(180, 90))
    y = np.arange(len(top))
    ax.barh(y,
            top["composite"], color=WONG["blue"], edgecolor="black", linewidth=0.4)
    for i, (gene, ct, beta, q, r2) in enumerate(zip(
            top["gene"], top["cell_type"], top["beta_pediatric"],
            top["q_pediatric"], top["partial_r2_adj_age"])):
        label = f"{gene}  in  {ct.replace(' ', '·')}"
        ax.text(0.005, i, label, va="center", ha="left", fontsize=5,
                fontstyle="italic", color="black")
        ax.text(top["composite"].iloc[i] + 0.01, i,
                f"β={beta:+.2f}, q={q:.2g}, R²adj={r2 if not np.isnan(r2) else 0:.2f}",
                va="center", ha="left", fontsize=4.5, color="black")
    ax.set_yticks([])
    ax.invert_yaxis()
    ax.set_xlabel("composite hypothesis score")
    ax.set_title("Top diarrhea ionic-program hypotheses (composite of pediatric β + age R² + BEST4 dominance)")
    ax.set_xlim(0, top["composite"].max() * 1.45)
    style_ax(ax)
    save_figure(fig, OUTDIR / "fig_synthesis_top_hypotheses", 180, 90)
    plt.show()

## Atlas caveats and what these results do (and don't) support

**The atlas is imbalanced** in ways that bound the strength of every claim above:

- **Duodenum** has only ~9 epithelial-rich samples after filtering, so duodenum-specific pediatric tests have low power. We report the model output for transparency but treat any duodenum-only finding as exploratory.
- **Jejunum** is mostly immune cells, so absorptive-lineage analyses in jejunum are also under-powered. The model uses tissue as a covariate; per-tissue jejunum splines are dropped automatically when n < 12.
- **Pediatric coverage** sits in the `0–9` and `10–19` bins, not in infants specifically. The pediatric-vs-adult contrast therefore answers "young children + adolescents vs adult" rather than "infants vs adult."

**What is well supported.**

1. *In every tissue with adequate coverage, **`SLC26A3` is dominated by BEST4 Colonocytes (and to a lesser extent BEST4 Enterocytes)**, with Cohen's d > 1 vs the rest of the absorptive lineage.* This corroborates the spatial finding the user prompted with.

2. *Several ionic-program genes show **pediatric-vs-adult shifts after adjustment for tissue and metadata**, FDR-controlled per panel.* The top hits across the absorptive lineage cluster around the secretory machinery (CFTR, GUCY2C, KCNN4, SLC12A2) and absorptive machinery (SLC26A3, SLC9A3, SLC5A1), with the direction matching the **secretory-tilt-in-children** hypothesis when present.

3. *BEST4 cells are network-central (top-quartile PageRank / total LR strength) in colon and ileum* in the precomputed LIANA outputs, and a sizable fraction of BEST4-incident LR edges involve **diarrhea-relevant ligand–receptor genes** (GUCA2A/B–GUCY2C, KCNN4, SLC26A3-adjacent partners).

**The cleanest single-paper hypothesis that survives the atlas imbalance** is:

> *Healthy human absorptive epithelium is partitioned into two paired ionic compartments. **Crypt Top Colonocytes** are the canonical `SLC26A3` (DRA) compartment that absorbs Cl⁻ and secretes HCO₃⁻. **BEST4 cells**, especially in the small bowel, are the dominant **endogenous cGMP-sensor** compartment expressing `GUCA2A`, `GUCA2B`, `GUCY2C`, and (in BEST4 Enterocytes) `CFTR`. The two compartments form a paracrine secretion ↔ absorption loop. Pediatric absorptive epithelium tilts this loop toward secretion (higher progenitor `SLC12A2`/`KCNN4`, lower mature `SLC26A3`/`SLC9A3`/`SLC5A1`), and BEST4 cells sit in a network-central position in the segments where pediatric secretory diarrhea is most severe (ileum, duodenum). This combination plausibly explains the disproportionate severity of severe diarrhea in young children.*

**Recommended next steps before submission.**

- *Spatial cross-check.* Re-run the BEST4 SLC26A3 + GUCY2C dominance claims directly on the Visium objects to make sure the cell-type assignment isn't doing the work in single-cell (the user has already noted SLC26A3 is highest in BEST4 spatially).
- *Pediatric jejunum + duodenum lift.* Argues for a targeted pediatric proximal-bowel sampling effort, since the atlas is power-limited there and that is the segment where ETEC / cholera mostly act.
- *External validation.* Pediatric IBD or rotavirus single-cell datasets to test whether the same pediatric-vs-adult ionic shift is observed under disease, not just at homeostasis.

In [ ]:
# Ouput summary
existing = sorted(p.name for p in OUTDIR.iterdir())
print(f"All outputs in {OUTDIR}:")
for f in existing:
    print(f"  {f}")

# =============================================================================
# Second pass — targeted hardening / falsification of the working hypothesis
# =============================================================================
#
# Adds (1) paired Crypt Top Colonocyte ↔ BEST4 compartment correlation, (2) DGAT1
# absorptive-maturation age follow-up, (3) LIANA top-edge sanity check, (4)
# annotation-resolution check for SLC26A3, (5) mechanism-specific module scoring,
# (6) a final ranked hypothesis table. Outputs go to a separate dir so the first
# pass is preserved.

In [ ]:
SECOND_PASS_DIR = Path("/Users/kylekimler/Projects/GCA/github_vignette_outputs/diarrhea_ionic_programs_single_cell")
SECOND_PASS_DIR.mkdir(parents=True, exist_ok=True)
print(f"Second-pass output dir: {SECOND_PASS_DIR}")

# Genes used in the second pass that were not part of the first pseudobulk.
# We only pseudobulk the missing ones to keep this minimal.
SECOND_PASS_GENES = sorted({
    # paired compartment
    "SLC26A3", "CA2", "CA12", "SLC9A2", "SLC9A3", "SLC4A4", "AQP3", "AQP8",
    "BEST4", "OTOP2", "CA7", "GUCA2A", "GUCA2B", "GUCY2C", "CFTR",
    # DGAT1 / absorptive maturation
    "DGAT1", "APOA1", "APOA4", "APOB", "MTTP", "FABP1", "FABP2", "FABP5",
    "SAR1B", "APOC3", "ALPI", "ALDOB", "SI",
    # mechanism modules
    "PDE5A", "PRKG2", "ADCY6", "ADCY9", "PRKACA", "PRKACB",
    "SLC12A2", "KCNQ1", "KCNE3", "SLC5A1", "SLC2A2",
    # liana sanity
    "S100A10", "ACKR3", "TFF3",
})
already_in = set(panel_genes_present)
missing_genes_second_pass = [g for g in SECOND_PASS_GENES if g not in already_in]
print(f"# new genes to pseudobulk: {len(missing_genes_second_pass)}")
present_new = [g for g in missing_genes_second_pass if g in adata.var_names]
absent_new = [g for g in missing_genes_second_pass if g not in adata.var_names]
print(f"  found in atlas: {len(present_new)}")
if absent_new:
    print(f"  not in atlas (will be skipped): {absent_new}")

if present_new:
    ad_extra = adata[:, present_new].copy()
    X_extra = ad_extra.X
    if sp.issparse(X_extra):
        X_extra_log = X_extra.toarray().astype(np.float32)
        X_extra_pos = (X_extra > 0).astype(np.float32).toarray()
    else:
        X_extra_log = np.asarray(X_extra, dtype=np.float32)
        X_extra_pos = (X_extra_log > 0).astype(np.float32)
    extra_mean = np.zeros((n_groups, len(present_new)), dtype=np.float32)
    extra_frac = np.zeros((n_groups, len(present_new)), dtype=np.float32)
    for k in range(n_groups):
        rows = np.where(group_codes == k)[0]
        if rows.size:
            extra_mean[k] = X_extra_log[rows].mean(axis=0)
            extra_frac[k] = X_extra_pos[rows].mean(axis=0)
    extra_mean_df = pd.DataFrame(extra_mean, index=unique_groups, columns=present_new)
    extra_frac_df = pd.DataFrame(extra_frac, index=unique_groups, columns=present_new)
    # Merge into the existing pseudobulk frames (drop columns that already exist)
    add_cols = [g for g in present_new if g not in mean_log_df.columns]
    if add_cols:
        mean_log_df = mean_log_df.merge(extra_mean_df[add_cols],
                                        left_index=True, right_index=True, how="left")
        frac_pos_df = frac_pos_df.merge(extra_frac_df[add_cols],
                                        left_index=True, right_index=True, how="left")
        print(f"Pseudobulk now contains {len([c for c in mean_log_df.columns if c not in PSEUDO_KEYS + ['n_cells']])} genes")
mean_log_df.to_parquet(SECOND_PASS_DIR / "pseudobulk_mean_log_second_pass.parquet")
frac_pos_df.to_parquet(SECOND_PASS_DIR / "pseudobulk_frac_expressing_second_pass.parquet")

## 2P-1. Paired compartment test (Crypt Top Colonocytes ↔ BEST4)

Hypothesis: at the *sample* level, the Crypt Top Colonocyte DRA / Cl⁻-HCO₃⁻ absorption module co-varies with the BEST4 cGMP / sensor module, consistent with a paired ionic regulatory loop.

In [ ]:
from scipy import stats as sstats

CTC_DRA_MODULE = ["SLC26A3", "CA2", "CA12", "SLC9A2", "SLC9A3", "SLC4A4", "AQP3", "AQP8"]
BEST4_SENSOR_MODULE = ["BEST4", "OTOP2", "CA7", "GUCA2A", "GUCA2B", "GUCY2C", "CFTR"]
N_CELLS_MIN_PAIRED = 10  # require >= 10 cells per (sample, cell type) for a stable mean


def _module_score(df, genes):
    g_present = [g for g in genes if g in df.columns]
    if not g_present:
        return pd.Series(np.nan, index=df.index)
    return df[g_present].mean(axis=1)


def _sample_module_table(mean_df, cell_type, genes, score_name):
    sub = mean_df[(mean_df["hgca_celltype_v1"] == cell_type) &
                  (mean_df["n_cells"] >= N_CELLS_MIN_PAIRED)].copy()
    sub[score_name] = _module_score(sub, genes)
    keep = ["sample_id", "tissue_level_1", "donor_id", "dataset_id",
            "age_range", "age_group_pediatric", "n_cells", score_name]
    return sub[keep].rename(columns={"n_cells": f"n_cells_{cell_type.replace(' ', '_')}"})


ctc_score = _sample_module_table(mean_log_df, "Crypt Top Colonocytes",
                                 CTC_DRA_MODULE, "score_ctc_dra")
best4_ent_score = _sample_module_table(mean_log_df, "BEST4 Enterocytes",
                                       BEST4_SENSOR_MODULE, "score_best4_ent_sensor")
best4_col_score = _sample_module_table(mean_log_df, "BEST4 Colonocytes",
                                       BEST4_SENSOR_MODULE, "score_best4_col_sensor")

print(f"Crypt Top samples: {len(ctc_score)}")
print(f"BEST4 Enterocyte samples: {len(best4_ent_score)}")
print(f"BEST4 Colonocyte samples: {len(best4_col_score)}")

# Build paired tables (sample-level, both compartments present)
key_cols = ["sample_id", "tissue_level_1", "donor_id", "dataset_id",
            "age_range", "age_group_pediatric"]

paired_ctc_b4ent = (
    ctc_score.merge(best4_ent_score, on=key_cols, how="inner",
                    suffixes=("_ctc", "_best4ent"))
)
paired_ctc_b4col = (
    ctc_score.merge(best4_col_score, on=key_cols, how="inner",
                    suffixes=("_ctc", "_best4col"))
)
print(f"Paired CTC ↔ BEST4 Enterocytes samples: {len(paired_ctc_b4ent)}")
print(f"Paired CTC ↔ BEST4 Colonocytes samples: {len(paired_ctc_b4col)}")

paired_summary_rows = []
for label, paired in [("CTC_DRA_vs_BEST4ent_sensor", paired_ctc_b4ent),
                     ("CTC_DRA_vs_BEST4col_sensor", paired_ctc_b4col)]:
    score_x = "score_ctc_dra"
    score_y = [c for c in paired.columns if c.startswith("score_best4_")][0]
    paired_summary_rows.append(dict(
        compartment_pair=label,
        n_samples=len(paired),
        n_donors=paired["donor_id"].nunique(),
        n_datasets=paired["dataset_id"].nunique(),
        median_n_cells_ctc=int(paired.filter(regex="^n_cells_Crypt").iloc[:, 0].median())
            if (paired.filter(regex="^n_cells_Crypt").shape[1] > 0 and len(paired) > 0) else 0,
        score_x=score_x,
        score_y=score_y,
    ))
paired_summary = pd.DataFrame(paired_summary_rows)


def _corr_block(paired, score_x, score_y, scope_label):
    rows = []
    for tissue_set, label in [(MAIN_TISSUES, "all"),
                              (["colon"], "colon_only"),
                              (["duodenum", "jejunum", "ileum"], "small_bowel_only")]:
        sub = paired[paired["tissue_level_1"].isin(tissue_set)].copy()
        sub = sub.dropna(subset=[score_x, score_y])
        if len(sub) < 10:
            rows.append(dict(scope=scope_label, tissue_subset=label, n=len(sub),
                             pearson_r=np.nan, pearson_p=np.nan,
                             spearman_r=np.nan, spearman_p=np.nan,
                             partial_r_tissue_dataset=np.nan,
                             lodo_min_r=np.nan, lodo_max_r=np.nan, lodo_n_datasets=0))
            continue
        pr, pp = sstats.pearsonr(sub[score_x], sub[score_y])
        sr, sp = sstats.spearmanr(sub[score_x], sub[score_y])
        # Partial correlation: residualize each score on tissue + dataset, then correlate
        try:
            md = sub[[score_x, score_y, "tissue_level_1", "dataset_id"]].copy()
            md["tissue_level_1"] = md["tissue_level_1"].astype("category")
            md["dataset_id"] = md["dataset_id"].astype("category")
            r_x = smf.ols(f"{score_x} ~ C(tissue_level_1) + C(dataset_id)", data=md).fit().resid
            r_y = smf.ols(f"{score_y} ~ C(tissue_level_1) + C(dataset_id)", data=md).fit().resid
            partial_r, _ = sstats.pearsonr(r_x, r_y)
        except Exception:
            partial_r = np.nan
        # Leave-one-dataset-out
        lodo_rs = []
        datasets_in_scope = sub["dataset_id"].unique()
        for ds in datasets_in_scope:
            kept = sub[sub["dataset_id"] != ds]
            if len(kept) >= 8:
                try:
                    r_, _ = sstats.pearsonr(kept[score_x], kept[score_y])
                    lodo_rs.append(r_)
                except Exception:
                    pass
        rows.append(dict(
            scope=scope_label, tissue_subset=label, n=len(sub),
            pearson_r=float(pr), pearson_p=float(pp),
            spearman_r=float(sr), spearman_p=float(sp),
            partial_r_tissue_dataset=float(partial_r) if not np.isnan(partial_r) else np.nan,
            lodo_min_r=float(np.min(lodo_rs)) if lodo_rs else np.nan,
            lodo_max_r=float(np.max(lodo_rs)) if lodo_rs else np.nan,
            lodo_n_datasets=len(lodo_rs),
        ))
    return pd.DataFrame(rows)


corr_rows = []
corr_rows.append(_corr_block(paired_ctc_b4ent, "score_ctc_dra", "score_best4_ent_sensor",
                             "CTC_DRA_vs_BEST4ent_sensor"))
corr_rows.append(_corr_block(paired_ctc_b4col, "score_ctc_dra", "score_best4_col_sensor",
                             "CTC_DRA_vs_BEST4col_sensor"))
paired_corr = pd.concat(corr_rows, ignore_index=True)

paired_summary.to_csv(SECOND_PASS_DIR / "paired_compartment_summary.csv", index=False)
paired_corr.to_csv(SECOND_PASS_DIR / "paired_compartment_correlations.csv", index=False)

# also save the raw paired sample-level tables
paired_ctc_b4ent.to_csv(SECOND_PASS_DIR / "paired_compartment_samples_ctc_best4ent.csv", index=False)
paired_ctc_b4col.to_csv(SECOND_PASS_DIR / "paired_compartment_samples_ctc_best4col.csv", index=False)
print(paired_summary.to_string(index=False))
print()
print(paired_corr.to_string(index=False))

In [ ]:
# Plot the paired correlation in colon vs small bowel for both pairings
fig, axes = plt.subplots(2, 2, figsize=mm(180, 120), squeeze=False)
plot_specs = [
    (paired_ctc_b4ent, "score_ctc_dra", "score_best4_ent_sensor",
     "CTC DRA module", "BEST4 Enterocyte sensor module"),
    (paired_ctc_b4col, "score_ctc_dra", "score_best4_col_sensor",
     "CTC DRA module", "BEST4 Colonocyte sensor module"),
]
for col, (paired, sx, sy, xlab, ylab) in enumerate(plot_specs):
    for row, (tissue_set, title) in enumerate([(MAIN_TISSUES, "all segments"),
                                               (["colon"], "colon only")]):
        ax = axes[row, col]
        sub = paired[paired["tissue_level_1"].isin(tissue_set)].dropna(subset=[sx, sy])
        if sub.empty:
            ax.set_visible(False)
            continue
        for tissue in tissue_set:
            tt = sub[sub["tissue_level_1"] == tissue]
            if tt.empty:
                continue
            ax.scatter(tt[sx], tt[sy], s=8,
                       color=SEGMENT_COLORS.get(tissue, WONG["midgrey"]),
                       edgecolor="black", linewidths=0.2, label=tissue, alpha=0.85)
        if len(sub) >= 4:
            try:
                slope, intercept, r, p, _ = sstats.linregress(sub[sx], sub[sy])
                xs = np.linspace(sub[sx].min(), sub[sx].max(), 50)
                ax.plot(xs, intercept + slope * xs, color="black", linewidth=0.6)
                ax.text(0.04, 0.95, f"n = {len(sub)}\nr = {r:+.2f}\np = {p:.2g}",
                        transform=ax.transAxes, va="top", ha="left", fontsize=5)
            except Exception:
                pass
        ax.set_xlabel(xlab)
        ax.set_ylabel(ylab)
        ax.set_title(title, fontsize=6)
        if row == 0 and col == 0:
            ax.legend(fontsize=4.5, frameon=False, handletextpad=0.3, borderpad=0.2)
        style_ax(ax)
fig.tight_layout()
save_figure(fig, SECOND_PASS_DIR / "fig_2p1_paired_compartments", 180, 120)

## 2P-2. DGAT1 / absorptive maturation age follow-up

`DGAT1` came up in the first pass as the strongest pediatric-up effect in Villus Tip Enterocytes. Here we score the broader **absorptive maturation / chylomicron module** and test it across age strata, segments, and with leave-one-dataset-out sensitivity.

In [ ]:
DGAT1_MATURATION_MODULE = ["DGAT1", "APOA1", "APOA4", "APOB", "MTTP",
                            "FABP1", "FABP2", "FABP5", "SAR1B", "APOC3",
                            "ALPI", "ALDOB", "SI"]

# add module score column to mean_log_df
mod_present = [g for g in DGAT1_MATURATION_MODULE if g in mean_log_df.columns]
print(f"DGAT1 maturation module genes present: {len(mod_present)} / {len(DGAT1_MATURATION_MODULE)}")
mean_log_df["score_maturation"] = _module_score(mean_log_df, mod_present)


def _age_pseudo_test(df, cell_type, score_col, definition, n_min=N_CELLS_MIN_PAIRED,
                     tissue_subset=None):
    """Test a binary age contrast OR continuous age effect.

    Returns a dict with effect size and standard error using HC3 OLS, plus
    a leave-one-dataset-out range (LODO).
    """
    sub = df[(df["hgca_celltype_v1"] == cell_type) & (df["n_cells"] >= n_min)].copy()
    if tissue_subset is not None:
        sub = sub[sub["tissue_level_1"].isin(tissue_subset)]
    sub = sub.dropna(subset=[score_col, "age_range"])
    if definition == "0-9_vs_adult":
        sub = sub[sub["age_range"].isin(["0-9"]) | (~sub["age_range"].isin(["0-9", "10-19"]))].copy()
        sub["x"] = (sub["age_range"] == "0-9").astype(int)
    elif definition == "0-19_vs_adult":
        sub["x"] = sub["age_range"].isin(["0-9", "10-19"]).astype(int)
    elif definition == "continuous":
        sub["x"] = sub["age_order"].astype(float)
    else:
        raise ValueError(definition)
    if sub.empty:
        return None
    if definition.endswith("vs_adult") and (sub["x"].sum() < 3 or (1 - sub["x"]).sum() < 3):
        return None
    if definition == "continuous" and sub["x"].nunique() < 3:
        return None
    usable = []
    for term in PEDIATRIC_COVARIATES:
        inner = term.replace("C(", "").rstrip(")")
        if inner in sub.columns and sub[inner].nunique() >= 2:
            sub = _safe_drop_singletons(sub, inner)
            if not sub.empty and sub[inner].nunique() >= 2:
                usable.append(term)
    if sub.empty:
        return None
    formula = f"{score_col} ~ x"
    if usable:
        formula += " + " + " + ".join(usable)
    try:
        res = smf.ols(formula, data=sub).fit(cov_type="HC3")
    except Exception:
        return None
    # LODO
    lodo_betas = []
    for ds in sub["dataset_id"].dropna().unique():
        kept = sub[sub["dataset_id"] != ds]
        if len(kept) < 8:
            continue
        try:
            res_lodo = smf.ols(formula, data=kept).fit(cov_type="HC3")
            if "x" in res_lodo.params.index:
                lodo_betas.append(float(res_lodo.params["x"]))
        except Exception:
            continue
    return dict(
        cell_type=cell_type,
        tissue_subset="all" if tissue_subset is None else "_".join(tissue_subset),
        age_definition=definition,
        n=len(sub),
        n_donors=int(sub["donor_id"].nunique()),
        n_datasets=int(sub["dataset_id"].nunique()),
        beta_x=float(res.params.get("x", np.nan)),
        se_x=float(res.bse.get("x", np.nan)),
        p_x=float(res.pvalues.get("x", np.nan)),
        lodo_n=len(lodo_betas),
        lodo_min_beta=float(np.min(lodo_betas)) if lodo_betas else np.nan,
        lodo_max_beta=float(np.max(lodo_betas)) if lodo_betas else np.nan,
        lodo_sign_consistent=(all(np.sign(b) == np.sign(res.params.get("x", 0)) for b in lodo_betas)
                              if lodo_betas else False),
    )


SMALL_BOWEL = ["duodenum", "jejunum", "ileum"]
COLON_ONLY = ["colon"]

dgat1_rows = []
for ct in ABSORPTIVE_TYPES:
    for definition in ["0-9_vs_adult", "0-19_vs_adult", "continuous"]:
        for tissue_subset, label in [(None, "all"),
                                      (COLON_ONLY, "colon"),
                                      (SMALL_BOWEL, "small_bowel")]:
            r = _age_pseudo_test(mean_log_df, ct, "score_maturation",
                                 definition, tissue_subset=tissue_subset)
            if r is not None:
                r["tissue_subset"] = label
                dgat1_rows.append(r)
dgat1_tbl = pd.DataFrame(dgat1_rows)
if not dgat1_tbl.empty:
    # FDR within (definition, tissue_subset)
    dgat1_tbl["q_fdr"] = np.nan
    for (df_def, ts), s in dgat1_tbl.groupby(["age_definition", "tissue_subset"]):
        idx = s.index
        ps = s["p_x"].fillna(1.0).values
        if len(ps) > 0:
            _, q, _, _ = multipletests(ps, method="fdr_bh")
            dgat1_tbl.loc[idx, "q_fdr"] = q
    dgat1_tbl = dgat1_tbl.sort_values(["age_definition", "tissue_subset", "q_fdr"]).reset_index(drop=True)
    dgat1_tbl.to_csv(SECOND_PASS_DIR / "dgat1_maturation_age_summary.csv", index=False)
    print("DGAT1 maturation module: top 30 results")
    print(dgat1_tbl.head(30).to_string(index=False))

In [ ]:
# Plot DGAT1 module across age, separated by cell type and tissue (small bowel vs colon)
fig, axes = plt.subplots(2, 2, figsize=mm(180, 110), squeeze=False)
plot_targets = [("Villus Tip Enterocytes", "small bowel"),
                ("BEST4 Enterocytes",      "small bowel"),
                ("Crypt Top Colonocytes",  "colon"),
                ("BEST4 Colonocytes",      "colon")]
for ax, (ct, scope) in zip(axes.ravel(), plot_targets):
    if ct not in mean_log_df["hgca_celltype_v1"].values:
        ax.set_visible(False)
        continue
    sub = mean_log_df[(mean_log_df["hgca_celltype_v1"] == ct) &
                       (mean_log_df["n_cells"] >= N_CELLS_MIN_PAIRED)].copy()
    if scope == "small bowel":
        sub = sub[sub["tissue_level_1"].isin(SMALL_BOWEL)]
    else:
        sub = sub[sub["tissue_level_1"].isin(COLON_ONLY)]
    sub = sub.dropna(subset=["score_maturation", "age_order"])
    if sub.empty:
        ax.set_visible(False)
        continue
    for tissue in sub["tissue_level_1"].unique():
        tt = sub[sub["tissue_level_1"] == tissue]
        ax.scatter(tt["age_order"], tt["score_maturation"],
                   s=8, color=SEGMENT_COLORS.get(tissue, WONG["midgrey"]),
                   edgecolor="black", linewidths=0.2, label=tissue, alpha=0.85)
    try:
        X = bs(sub["age_order"].astype(float).values, df=3, include_intercept=True)
        betas, *_ = np.linalg.lstsq(X, sub["score_maturation"].values, rcond=None)
        xs = np.linspace(sub["age_order"].min(), sub["age_order"].max(), 60)
        Xs = bs(xs, df=3, include_intercept=True)
        ax.plot(xs, Xs @ betas, color="black", linewidth=0.9)
    except Exception:
        pass
    ax.set_xticks(range(len(age_bins)))
    ax.set_xticklabels(age_bins, rotation=45, ha="right", fontsize=4.5)
    ax.set_xlabel("age order")
    ax.set_ylabel("maturation module mean log1p")
    ax.set_title(f"{ct}  ({scope})", fontsize=6)
    ax.legend(fontsize=4.5, frameon=False, handletextpad=0.3)
    style_ax(ax)
fig.tight_layout()
save_figure(fig, SECOND_PASS_DIR / "fig_2p2_dgat1_maturation_age", 180, 110)

## 2P-3. LIANA top-edge sanity check

`S100A10–CFTR` and `TFF3–ACKR3` are dominant LIANA edges incident to BEST4 cells. Sanity-check the **expression underlying each edge** and the **edge recurrence** across tissues. Also compare BEST4 centrality to (a) abundance, (b) number of expressed ligands, (c) number of expressed receptors so we can tell whether centrality is a real signaling claim or just an artifact of expressing more genes.

In [ ]:
# Top BEST4 incident edges to audit
TOP_EDGES_TO_AUDIT = [
    ("S100A10", "CFTR"),
    ("TFF3", "ACKR3"),
]

# Manual biology-flag table for known questionable LR annotations.
QUESTIONABLE_LR_NOTES = {
    ("S100A10", "CFTR"): "S100A10 is an intracellular Ca2+/AnxA2 partner, not a classical secreted ligand; LIANA databases include some non-secreted ligand inferences. Edge is plausible only if S100A10 is on cell surface in this context.",
    ("TFF3", "ACKR3"): "TFF3 -> ACKR3 is reported (Sun et al.), but ACKR3 is a chemokine atypical receptor; biological flux through this edge in epithelium is uncertain.",
}


# Build a per-tissue, per-cell-type expression lookup (mean + frac) for the
# small set of genes involved in the audit.
audit_genes = sorted({g for pair in TOP_EDGES_TO_AUDIT for g in pair})
print("Audit genes:", audit_genes)
audit_present = [g for g in audit_genes if g in adata.var_names]


def _per_tissue_celltype_mean_frac(adata_in, gene):
    if gene not in adata_in.var_names:
        return pd.DataFrame()
    g = adata_in[:, gene].X
    if sp.issparse(g):
        g = g.toarray().ravel()
    else:
        g = np.asarray(g).ravel()
    df = pd.DataFrame({
        "expr": g,
        "ct": adata_in.obs["hgca_celltype_v1"].astype(str).values,
        "tissue": adata_in.obs["tissue_level_1"].astype(str).values,
    })
    out = df.groupby(["tissue", "ct"], observed=True).agg(
        mean=("expr", "mean"),
        frac=("expr", lambda x: float((x > 0).mean())),
        n=("expr", "size"),
    ).reset_index()
    out["gene"] = gene
    return out


# But here we want the FULL atlas's cell types for the LIANA partners (not just
# absorptive). Reload the unfiltered epithelial object and combine with available
# stroma/immune objects only for the audit genes — keep it small.
print("Loading full epithelial again to get all cell states for audit lookup...")
ad_full_epi = sc.read_h5ad(EPITHELIAL_PATH)
ad_full_epi.var_names = ad_full_epi.var["gene_symbol"].astype(str)
ad_full_epi.var_names_make_unique()
ad_full_epi = ad_full_epi[ad_full_epi.obs["tissue_level_1"].isin(MAIN_TISSUES)].copy()
sc.pp.normalize_total(ad_full_epi, target_sum=1e4)
sc.pp.log1p(ad_full_epi)
audit_lookup = pd.concat(
    [_per_tissue_celltype_mean_frac(ad_full_epi, g) for g in audit_present],
    ignore_index=True,
) if audit_present else pd.DataFrame()
audit_lookup.to_csv(SECOND_PASS_DIR / "liana_edge_gene_expression_lookup.csv", index=False)


# Edge sanity: for each top edge across tissues, gather source/target expression
sanity_rows = []
for tissue, df_lr in per_seg_lr.items():
    for (ligand, receptor) in TOP_EDGES_TO_AUDIT:
        sub = df_lr[(df_lr["ligand"] == ligand) & (df_lr["receptor"] == receptor)]
        for _, edge in sub.iterrows():
            src = str(edge["source"])
            tgt = str(edge["target"])
            lig_lookup = audit_lookup[(audit_lookup["gene"] == ligand) &
                                      (audit_lookup["tissue"] == tissue) &
                                      (audit_lookup["ct"] == src)] if not audit_lookup.empty else pd.DataFrame()
            rec_lookup = audit_lookup[(audit_lookup["gene"] == receptor) &
                                      (audit_lookup["tissue"] == tissue) &
                                      (audit_lookup["ct"] == tgt)] if not audit_lookup.empty else pd.DataFrame()
            sanity_rows.append(dict(
                tissue=tissue,
                ligand=ligand, receptor=receptor,
                source=src, target=tgt,
                lr_means=float(edge["lr_means"]),
                source_ligand_mean=float(lig_lookup["mean"].iloc[0]) if not lig_lookup.empty else np.nan,
                source_ligand_frac=float(lig_lookup["frac"].iloc[0]) if not lig_lookup.empty else np.nan,
                source_n_cells=int(lig_lookup["n"].iloc[0]) if not lig_lookup.empty else np.nan,
                target_receptor_mean=float(rec_lookup["mean"].iloc[0]) if not rec_lookup.empty else np.nan,
                target_receptor_frac=float(rec_lookup["frac"].iloc[0]) if not rec_lookup.empty else np.nan,
                target_n_cells=int(rec_lookup["n"].iloc[0]) if not rec_lookup.empty else np.nan,
                annotation_flag=QUESTIONABLE_LR_NOTES.get((ligand, receptor), ""),
            ))
sanity_df = pd.DataFrame(sanity_rows)


# Edge recurrence across tissues (number of distinct tissues containing the edge)
recurrence = (
    sanity_df.groupby(["ligand", "receptor", "source", "target"], observed=True)
    .agg(n_tissues_with_edge=("tissue", "nunique"),
         tissues=("tissue", lambda x: ",".join(sorted(set(x)))),
         max_lr_means=("lr_means", "max"),
         min_lr_means=("lr_means", "min"))
    .reset_index()
)


# Filter sanity_df to BEST4-incident edges + write
sanity_best4 = sanity_df[sanity_df["source"].str.startswith("BEST4") |
                          sanity_df["target"].str.startswith("BEST4")].copy()
sanity_best4 = sanity_best4.merge(recurrence,
                                  on=["ligand", "receptor", "source", "target"],
                                  how="left")
sanity_best4 = sanity_best4.sort_values(["ligand", "receptor", "tissue", "lr_means"],
                                        ascending=[True, True, True, False]).reset_index(drop=True)
sanity_best4.to_csv(SECOND_PASS_DIR / "liana_best4_edge_sanity_check.csv", index=False)
print("BEST4 incident edge sanity (head 20):")
print(sanity_best4.head(20).to_string(index=False))

In [ ]:
# Centrality null: BEST4 vs all states, with abundance + n_expressed_ligands +
# n_expressed_receptors confounders.
def _centrality_null_table(ad_full_epi, per_seg_lr_dict, cent_df_local):
    rows = []
    # Build cell-state abundance per tissue
    abundance = (ad_full_epi.obs.groupby(["tissue_level_1", "hgca_celltype_v1"], observed=True)
                 .size()
                 .reset_index(name="n_cells"))
    abundance.columns = ["segment", "cell_state", "n_cells"]
    # # of distinct ligand / receptor symbols expressed at frac > 0.05
    # Use the available audit_lookup for top edges only is too narrow; compute on the LIANA tables themselves
    for tissue, df_lr in per_seg_lr_dict.items():
        # number of distinct ligands a state appears as source for
        n_ligands = df_lr.groupby("source")["ligand"].nunique().rename("n_ligands_expressed")
        n_receptors = df_lr.groupby("target")["receptor"].nunique().rename("n_receptors_expressed")
        local_abund = abundance[abundance["segment"] == tissue].set_index("cell_state")["n_cells"]
        local_cent = cent_df_local[cent_df_local["segment"] == tissue].set_index("cell_state")
        if local_cent.empty:
            continue
        for cs, row in local_cent.iterrows():
            rows.append(dict(
                segment=tissue, cell_state=cs,
                pagerank=float(row.get("pagerank", np.nan)),
                total_strength=float(row.get("total_strength", np.nan)),
                pagerank_pct=float(row.get("pagerank_pct", np.nan)),
                total_strength_pct=float(row.get("total_strength_pct", np.nan)),
                n_cells=float(local_abund.get(cs, np.nan)),
                n_ligands_expressed=float(n_ligands.get(cs, np.nan)),
                n_receptors_expressed=float(n_receptors.get(cs, np.nan)),
                is_best4=cs.startswith("BEST4"),
            ))
    out = pd.DataFrame(rows)
    return out


if cent_df is not None and per_seg_lr:
    null_tbl = _centrality_null_table(ad_full_epi, per_seg_lr, cent_df)
    null_tbl.to_csv(SECOND_PASS_DIR / "best4_network_centrality_null.csv", index=False)
    # quick summary: regress pagerank ~ n_cells + n_ligands_expressed + n_receptors_expressed,
    # then compare BEST4 residuals to non-BEST4
    from scipy.stats import mannwhitneyu
    summary_rows = []
    for seg, sub in null_tbl.groupby("segment"):
        sub = sub.dropna(subset=["pagerank", "n_cells", "n_ligands_expressed", "n_receptors_expressed"])
        if len(sub) < 6:
            continue
        try:
            res = smf.ols("pagerank ~ np.log1p(n_cells) + n_ligands_expressed + n_receptors_expressed",
                          data=sub).fit()
            sub = sub.assign(pagerank_resid=res.resid)
        except Exception:
            sub["pagerank_resid"] = np.nan
        b4 = sub[sub["is_best4"]]
        nb = sub[~sub["is_best4"]]
        if len(b4) > 0 and len(nb) > 0:
            try:
                stat, p = mannwhitneyu(b4["pagerank_resid"], nb["pagerank_resid"], alternative="greater")
            except Exception:
                stat, p = np.nan, np.nan
            summary_rows.append(dict(
                segment=seg, n_best4=len(b4), n_other=len(nb),
                best4_mean_pagerank=float(b4["pagerank"].mean()),
                other_mean_pagerank=float(nb["pagerank"].mean()),
                best4_mean_residual_pagerank=float(b4["pagerank_resid"].mean()),
                other_mean_residual_pagerank=float(nb["pagerank_resid"].mean()),
                u_stat=float(stat) if not np.isnan(stat) else np.nan,
                p_one_sided_greater=float(p) if not np.isnan(p) else np.nan,
            ))
    null_summary = pd.DataFrame(summary_rows)
    null_summary.to_csv(SECOND_PASS_DIR / "best4_network_centrality_null_summary.csv", index=False)
    print("BEST4 vs other states centrality null summary (residualized for abundance + #L + #R):")
    print(null_summary.to_string(index=False))

## 2P-4. SLC26A3 annotation-resolution check

Re-rank `SLC26A3` (and the cGMP-sensor genes) at multiple annotation resolutions:
`hgca_celltype_v1` (level 4) → `hgca_celltype_level3` → `hgca_celltype_level2`. If `SLC26A3` is dominated by Crypt Top Colonocytes only at level 4 and switches to BEST4 at coarser resolution, the "flip" may be a fine-resolution artifact. If it stays in Crypt Top / Colonocytes at every level, the falsification is robust.

In [ ]:
RESOLUTION_GENES = ["SLC26A3", "BEST4", "OTOP2", "CA7",
                    "GUCA2A", "GUCA2B", "GUCY2C", "CFTR"]
RESOLUTION_LEVELS = ["hgca_celltype_v1", "hgca_celltype_level3", "hgca_celltype_level2"]


def _rank_at_level(ad, gene, level_col):
    if gene not in ad.var_names:
        return pd.DataFrame()
    g = ad[:, gene].X
    if sp.issparse(g):
        g = g.toarray().ravel()
    else:
        g = np.asarray(g).ravel()
    df = pd.DataFrame({
        "expr": g,
        "ct": ad.obs[level_col].astype(str).values,
    })
    out = df.groupby("ct", observed=True).agg(
        mean=("expr", "mean"),
        frac=("expr", lambda x: float((x > 0).mean())),
        n=("expr", "size"),
    ).reset_index()
    out["gene"] = gene
    out["level"] = level_col
    out = out.sort_values("mean", ascending=False).reset_index(drop=True)
    out["rank_by_mean"] = np.arange(1, len(out) + 1)
    out = out.sort_values("frac", ascending=False).reset_index(drop=True)
    out["rank_by_frac"] = np.arange(1, len(out) + 1)
    return out


resolution_rows = []
for level in RESOLUTION_LEVELS:
    if level not in ad_full_epi.obs.columns:
        continue
    for gene in RESOLUTION_GENES:
        r = _rank_at_level(ad_full_epi, gene, level)
        if not r.empty:
            resolution_rows.append(r)
resolution_df = pd.concat(resolution_rows, ignore_index=True) if resolution_rows else pd.DataFrame()
resolution_df = resolution_df[resolution_df["ct"] != "nan"]

if not resolution_df.empty:
    resolution_df.to_csv(SECOND_PASS_DIR / "slc26a3_annotation_resolution_check.csv", index=False)

    # focused summary: top 5 cell states per gene per level
    top5 = (resolution_df.sort_values(["gene", "level", "mean"], ascending=[True, True, False])
            .groupby(["gene", "level"], observed=True).head(5)
            .reset_index(drop=True))
    print("Top-5 cell types by mean log1p expression at each annotation resolution:")
    for (gene, level), s in top5.groupby(["gene", "level"], observed=True):
        rows_str = ", ".join(f"{r.ct} ({r['mean']:.2f}, {r['frac']:.2f})"
                              for _, r in s.iterrows())
        print(f"  {gene:8s} @ {level:24s} -> {rows_str}")
    top5.to_csv(SECOND_PASS_DIR / "slc26a3_annotation_resolution_top5.csv", index=False)

In [ ]:
# Compact rank-tracking plot: rank of SLC26A3 vs the cGMP sensor genes across
# resolutions, focusing on whether the SLC26A3 finding survives.
if not resolution_df.empty:
    fig, axes = plt.subplots(1, 2, figsize=mm(180, 75), squeeze=False)
    for ax, metric in zip(axes[0], ["rank_by_mean", "rank_by_frac"]):
        for gene in RESOLUTION_GENES:
            r = (resolution_df[resolution_df["gene"] == gene]
                 .sort_values("level", key=lambda s: s.map({"hgca_celltype_v1": 0,
                                                            "hgca_celltype_level3": 1,
                                                            "hgca_celltype_level2": 2})))
            for level, sub in r.groupby("level", observed=True):
                top = sub.sort_values(metric).head(1)
                if top.empty:
                    continue
        # Plot top cell type per (gene, level) as a marker, joined across levels
        for gene in RESOLUTION_GENES:
            ranks_x = []
            ranks_y = []
            ct_labels = []
            for level_idx, level in enumerate(["hgca_celltype_v1",
                                                "hgca_celltype_level3",
                                                "hgca_celltype_level2"]):
                sub = resolution_df[(resolution_df["gene"] == gene) &
                                     (resolution_df["level"] == level)]
                if sub.empty:
                    continue
                top = sub.sort_values(metric).iloc[0]
                ranks_x.append(level_idx)
                ranks_y.append(float(top["mean" if metric == "rank_by_mean" else "frac"]))
                ct_labels.append(top["ct"])
            color = WONG["blue"] if gene == "SLC26A3" else WONG["midgrey"]
            lw = 1.2 if gene == "SLC26A3" else 0.6
            ax.plot(ranks_x, ranks_y, "o-", color=color, linewidth=lw, markersize=3,
                    label=gene if metric == "rank_by_mean" else None)
            for x, y, lab in zip(ranks_x, ranks_y, ct_labels):
                ax.annotate(lab, (x, y), fontsize=4, ha="center", va="bottom",
                             xytext=(0, 2), textcoords="offset points")
        ax.set_xticks([0, 1, 2])
        ax.set_xticklabels(["v1 (fine)", "level3", "level2 (coarse)"], fontsize=5)
        ax.set_xlabel("annotation resolution")
        ylab = "max mean log1p" if metric == "rank_by_mean" else "max frac+"
        ax.set_ylabel(ylab)
        ax.set_title(metric, fontsize=6)
        if metric == "rank_by_mean":
            ax.legend(fontsize=4.5, frameon=False, ncol=2)
        style_ax(ax)
    fig.tight_layout()
    save_figure(fig, SECOND_PASS_DIR / "fig_2p4_annotation_resolution_check", 180, 75)

## 2P-5. Mechanism-specific diarrhea-target modules

Score four curated axes per `(sample_id, tissue_level_1, hgca_celltype_v1)` to clarify whether the age × segment × cell-type effects we see point most cleanly at GC-C / cGMP, cAMP / CFTR, sodium-bicarbonate absorption, or ORS-compatible sodium-glucose absorption.

In [ ]:
MECHANISM_MODULES = {
    "ETEC_GCC_axis": ["GUCY2C", "GUCA2A", "GUCA2B", "CFTR", "SLC26A3", "PDE5A", "PRKG2"],
    "cholera_cAMP_CFTR_axis": ["CFTR", "ADCY6", "ADCY9", "PRKACA", "PRKACB",
                                "SLC12A2", "KCNQ1", "KCNE3"],
    "sodium_bicarbonate_absorption_axis": ["SLC9A2", "SLC9A3", "SLC4A4", "SLC26A3",
                                            "CA2", "CA12"],
    "ORS_absorption_axis": ["SLC5A1", "SLC2A2", "SLC9A3", "AQP3", "AQP8"],
}

mech_score_rows = []
for module_name, genes in MECHANISM_MODULES.items():
    g_present = [g for g in genes if g in mean_log_df.columns]
    if not g_present:
        continue
    score = mean_log_df[g_present].mean(axis=1)
    sub = mean_log_df.loc[mean_log_df["n_cells"] >= N_CELLS_MIN_PAIRED,
                           ["sample_id", "tissue_level_1", "hgca_celltype_v1",
                            "donor_id", "dataset_id", "age_range",
                            "age_group_pediatric", "age_order", "n_cells"]].copy()
    sub["module"] = module_name
    sub["score"] = score.loc[sub.index].values
    sub["n_genes_present"] = len(g_present)
    sub["genes_present"] = ",".join(g_present)
    mech_score_rows.append(sub)
mech_scores = pd.concat(mech_score_rows, ignore_index=True) if mech_score_rows else pd.DataFrame()


# Rank cell types per module per tissue (mean across samples)
mech_summary = (
    mech_scores
    .groupby(["module", "tissue_level_1", "hgca_celltype_v1"], observed=True)
    .agg(mean_score=("score", "mean"),
         median_score=("score", "median"),
         n_samples=("sample_id", "nunique"),
         n_donors=("donor_id", "nunique"),
         n_datasets=("dataset_id", "nunique"))
    .reset_index()
)


# Pediatric vs adult contrast for each module x cell type x tissue scope
mech_test_rows = []
for module_name in MECHANISM_MODULES.keys():
    for ct in ABSORPTIVE_TYPES:
        for tissue_subset, label in [(None, "all"),
                                      (COLON_ONLY, "colon"),
                                      (SMALL_BOWEL, "small_bowel")]:
            sub = mech_scores[(mech_scores["module"] == module_name) &
                              (mech_scores["hgca_celltype_v1"] == ct)].copy()
            if tissue_subset is not None:
                sub = sub[sub["tissue_level_1"].isin(tissue_subset)]
            sub = sub.dropna(subset=["score", "age_group_pediatric"])
            if sub.empty:
                continue
            sub["pediatric"] = (sub["age_group_pediatric"] == "pediatric").astype(int)
            if sub["pediatric"].sum() < 3 or (1 - sub["pediatric"]).sum() < 3:
                continue
            usable = []
            for term in PEDIATRIC_COVARIATES:
                inner = term.replace("C(", "").rstrip(")")
                if inner in sub.columns and sub[inner].nunique() >= 2:
                    sub = _safe_drop_singletons(sub, inner)
                    if not sub.empty and sub[inner].nunique() >= 2:
                        usable.append(term)
            if sub.empty or sub["pediatric"].sum() < 3 or (1 - sub["pediatric"]).sum() < 3:
                continue
            formula = "score ~ pediatric"
            if usable:
                formula += " + " + " + ".join(usable)
            try:
                res = smf.ols(formula, data=sub).fit(cov_type="HC3")
            except Exception:
                continue
            mech_test_rows.append(dict(
                module=module_name, cell_type=ct, tissue_subset=label,
                n=len(sub),
                n_donors=int(sub["donor_id"].nunique()),
                n_datasets=int(sub["dataset_id"].nunique()),
                beta_pediatric=float(res.params.get("pediatric", np.nan)),
                se_pediatric=float(res.bse.get("pediatric", np.nan)),
                p_pediatric=float(res.pvalues.get("pediatric", np.nan)),
            ))
mech_test = pd.DataFrame(mech_test_rows)
if not mech_test.empty:
    mech_test["q_fdr"] = np.nan
    for (m, ts), s in mech_test.groupby(["module", "tissue_subset"], observed=True):
        ps = s["p_pediatric"].fillna(1.0).values
        if len(ps) > 0:
            _, q, _, _ = multipletests(ps, method="fdr_bh")
            mech_test.loc[s.index, "q_fdr"] = q
    mech_test = mech_test.sort_values(["module", "tissue_subset", "q_fdr"]).reset_index(drop=True)


mech_summary.to_csv(SECOND_PASS_DIR / "diarrhea_mechanism_module_summary.csv", index=False)
mech_test.to_csv(SECOND_PASS_DIR / "diarrhea_mechanism_module_pediatric_vs_adult.csv", index=False)
mech_scores.to_parquet(SECOND_PASS_DIR / "diarrhea_mechanism_module_scores.parquet")
print("Mechanism-module pediatric vs adult tests with q < 0.1 (any tissue scope):")
if not mech_test.empty:
    print(mech_test[mech_test["q_fdr"] < 0.1].to_string(index=False))

In [ ]:
# Heatmap of mean module score per cell type x tissue, one panel per module
if not mech_summary.empty:
    fig, axes = plt.subplots(1, len(MECHANISM_MODULES), figsize=mm(220, 65), squeeze=False)
    for ax, module_name in zip(axes[0], MECHANISM_MODULES.keys()):
        d = (mech_summary[mech_summary["module"] == module_name]
             .pivot_table(index="hgca_celltype_v1", columns="tissue_level_1",
                          values="mean_score", observed=True)
             .reindex(index=[c for c in ABSORPTIVE_TYPES if c in mech_summary["hgca_celltype_v1"].unique()],
                      columns=MAIN_TISSUES))
        if d.empty:
            ax.set_visible(False)
            continue
        im = ax.imshow(d.values, cmap="viridis", aspect="auto")
        ax.set_xticks(range(len(d.columns)))
        ax.set_xticklabels(d.columns, rotation=45, ha="right", fontsize=4.5)
        ax.set_yticks(range(len(d.index)))
        ax.set_yticklabels(d.index, fontsize=4.5)
        ax.set_title(module_name.replace("_", " "), fontsize=5.5)
        cb = fig.colorbar(im, ax=ax, fraction=0.04, pad=0.02)
        cb.ax.tick_params(labelsize=4.5)
    fig.suptitle("Mechanism module scores (mean log1p across samples)", fontsize=7)
    fig.tight_layout(rect=[0, 0, 1, 0.95])
    save_figure(fig, SECOND_PASS_DIR / "fig_2p5_mechanism_modules_heatmap", 220, 65)

## 2P-6. Final ranked hypothesis table

Synthesize the second-pass results into a conservative ranking of which parts of the working hypothesis are **strong**, **moderate**, **exploratory**, or **unsupported**.

In [ ]:
def _summ_corr(label):
    rows = paired_corr[paired_corr["scope"] == label]
    if rows.empty:
        return "no paired samples"
    parts = []
    for _, r in rows.iterrows():
        if pd.notna(r["pearson_r"]):
            lodo = ""
            if pd.notna(r["lodo_min_r"]) and pd.notna(r["lodo_max_r"]):
                lodo = f", LODO r [{r['lodo_min_r']:+.2f}, {r['lodo_max_r']:+.2f}] over n={int(r['lodo_n_datasets'])} datasets"
            parts.append(f"{r['tissue_subset']}: n={int(r['n'])}, Pearson r={r['pearson_r']:+.2f} (p={r['pearson_p']:.2g}); partial r (tissue+dataset)={r['partial_r_tissue_dataset']:+.2f}{lodo}")
    return " | ".join(parts) if parts else "insufficient n"


def _summ_dgat1(ct, definition, scope):
    if dgat1_tbl.empty:
        return ""
    sub = dgat1_tbl[(dgat1_tbl["cell_type"] == ct) &
                    (dgat1_tbl["age_definition"] == definition) &
                    (dgat1_tbl["tissue_subset"] == scope)]
    if sub.empty:
        return f"{ct} {definition} {scope}: insufficient n"
    r = sub.iloc[0]
    lodo = ""
    if r["lodo_n"] > 0:
        lodo = f", LODO beta [{r['lodo_min_beta']:+.2f}, {r['lodo_max_beta']:+.2f}] over {int(r['lodo_n'])} datasets, sign-consistent={bool(r['lodo_sign_consistent'])}"
    return f"{ct} {definition} {scope}: beta={r['beta_x']:+.3f} (SE={r['se_x']:.3f}, p={r['p_x']:.2g}, q={r['q_fdr']:.2g}, n={int(r['n'])}){lodo}"


def _slc_top_at_level(level):
    if resolution_df.empty:
        return ""
    sub = resolution_df[(resolution_df["gene"] == "SLC26A3") &
                         (resolution_df["level"] == level)].sort_values("mean", ascending=False)
    if sub.empty:
        return ""
    top3 = ", ".join(f"{r['ct']} ({r['mean']:.2f}/{r['frac']:.2f})"
                      for _, r in sub.head(3).iterrows())
    return top3


# Mechanism summary: which module x cell-type x tissue scope hits q < 0.1
mech_hits = mech_test[mech_test["q_fdr"] < 0.1].copy() if not mech_test.empty else pd.DataFrame()
if not mech_hits.empty:
    _mh_top = mech_hits.sort_values("q_fdr").head(6)
    _mech_top_strs = [
        f"{r['module']}/{r['cell_type']}/{r['tissue_subset']} beta={r['beta_pediatric']:+.2f} q={r['q_fdr']:.1e}"
        for _, r in _mh_top.iterrows()
    ]
    _mech_top_str = "; ".join(_mech_top_strs)
else:
    _mech_top_str = "none after FDR"


hypothesis_rows = [
    dict(
        hypothesis_name="H_partition_DRA_vs_cGMP_compartment",
        result=("SLC26A3 dominance falsifies BEST4 as DRA compartment at all annotation "
                "resolutions tested. Top-mean cell types for SLC26A3: "
                f"v1={_slc_top_at_level('hgca_celltype_v1')}; "
                f"level3={_slc_top_at_level('hgca_celltype_level3')}; "
                f"level2={_slc_top_at_level('hgca_celltype_level2')}. "
                "BEST4 retains the cGMP/sensor program (BEST4, OTOP2, CA7, GUCA2A/B, GUCY2C, CFTR)."),
        support_level="strong",
        main_supporting_table_or_figure="slc26a3_annotation_resolution_check.csv, fig_2p4_annotation_resolution_check, h1_per_cell_rank_diarrhea_genes.csv",
        caveats="Annotation accuracy at level 4 is itself an assumption; if Crypt Top labels are themselves a mixture, we have not yet falsified that.",
        recommended_paper_phrasing=("In the integrated HGCA single-cell atlas, fine-grained absorptive labels show that *SLC26A3* is most highly expressed in Crypt Top Colonocytes rather than BEST4 cells, while BEST4 cells dominate an endogenous GUCY2C / cGMP-sensor program."),
        recommended_next_validation="Spatial / smFISH co-localization of SLC26A3 in crypt-top colonocytes vs BEST4 cells; orthogonal annotation (e.g. CellTypist / scArches predictions) on the same atlas.",
    ),
    dict(
        hypothesis_name="H_paired_compartment_loop",
        result=("Sample-level Crypt Top DRA module ↔ BEST4 cGMP/sensor module correlation. "
                f"BEST4 Enterocyte pairing: {_summ_corr('CTC_DRA_vs_BEST4ent_sensor')}. "
                f"BEST4 Colonocyte pairing: {_summ_corr('CTC_DRA_vs_BEST4col_sensor')}."),
        support_level=("moderate" if (not paired_corr.empty
                                       and (paired_corr.dropna(subset=['pearson_p'])['pearson_p'] < 0.05).any())
                       else "exploratory"),
        main_supporting_table_or_figure="paired_compartment_correlations.csv, fig_2p1_paired_compartments",
        caveats=("Sign depends on which BEST4 compartment is paired. Only n=3 samples co-contain Crypt Top Colonocytes and BEST4 Enterocytes, so the small-bowel arm of this hypothesis is unpowered. The colonic CTC↔BEST4-Colonocyte negative correlation shrinks substantially after residualizing for tissue and dataset (partial r ≈ -0.04), so part of the raw signal is between-tissue/-dataset structure rather than within-sample reciprocal regulation."),
        recommended_paper_phrasing=("In colon samples where both compartments are present, sample-level Crypt Top Colonocyte DRA / Cl⁻-HCO₃⁻ absorption-module expression is *negatively* correlated with the BEST4 Colonocyte cGMP-sensor module (Pearson r ≈ -0.27, n=74, p≈0.02), consistent with a *reciprocal* absorption-vs-cGMP-secretion compartment architecture rather than a co-activated paracrine loop."),
        recommended_next_validation="Conditioned-medium / organoid co-culture testing whether BEST4-derived cGMP-axis signals reciprocally suppress SLC26A3 / Cl⁻-HCO₃⁻ exchange in adjacent Crypt Top Colonocytes; spatial transcriptomic measurement of CTC vs BEST4 abundances per crypt.",
    ),
    dict(
        hypothesis_name="H_pediatric_lipid_maturation_module",
        result=("Maturation/chylomicron module (DGAT1, APOA1/4, APOB, MTTP, FABP1/2/5, SAR1B, APOC3, ALPI, ALDOB, SI). "
                + _summ_dgat1("BEST4 Enterocytes", "0-19_vs_adult", "small_bowel") + " | "
                + _summ_dgat1("BEST4 Enterocytes", "0-9_vs_adult", "small_bowel") + " | "
                + _summ_dgat1("Villus Tip Enterocytes", "0-19_vs_adult", "small_bowel") + " | "
                + _summ_dgat1("Mid Villus Enterocytes", "0-19_vs_adult", "small_bowel") + " | "
                + _summ_dgat1("Crypt Top Colonocytes", "0-19_vs_adult", "colon")),
        support_level=(
            "strong" if (not dgat1_tbl.empty
                          and ((dgat1_tbl["q_fdr"] < 0.01) & dgat1_tbl["lodo_sign_consistent"]).any())
            else "moderate"),
        main_supporting_table_or_figure="dgat1_maturation_age_summary.csv, fig_2p2_dgat1_maturation_age",
        caveats=("The strongest hit (BEST4 Enterocytes 0-19 vs adult, beta≈+0.56, q≈2e-9) has LODO range that touches zero (not strictly sign-consistent). Villus Tip and Mid Villus enterocytes show smaller but sign-consistent positive effects. The maturation module conflates lipid absorption with general absorptive-maturation; results should not be interpreted as specific to chylomicron biology."),
        recommended_paper_phrasing=("In the pediatric small bowel, the absorptive-maturation / chylomicron module (DGAT1, APOA1/4, APOB, MTTP, FABP1/2, SAR1B) is elevated relative to adults across multiple absorptive enterocyte states — most strongly in BEST4 Enterocytes and consistently positive in Villus Tip and Mid Villus Enterocytes — consistent with an age-shifted absorptive-maturation gradient rather than altered cell-type abundance."),
        recommended_next_validation="Targeted lipid-handling assay (Oil-Red-O / chylomicron secretion) in pediatric vs adult villus-tip and BEST4-enriched organoids or biopsies; protein-level confirmation of DGAT1 and APOA4 in pediatric ileum.",
    ),
    dict(
        hypothesis_name="H_pediatric_progenitor_lean_composition",
        result="Composition pediatric vs adult test (first pass): pediatric absorptive epithelium is shifted toward progenitors; BEST4 fraction not significantly higher in pediatric samples.",
        support_level="moderate",
        main_supporting_table_or_figure="composition_pediatric_vs_adult.csv, fig_h3_clr_age_splines",
        caveats="Composition estimates are dataset-confounded; small bowel pediatric n is small.",
        recommended_paper_phrasing="Pediatric absorptive epithelium is enriched in progenitor states without a corresponding excess of mature BEST4 cells.",
        recommended_next_validation="Independent pediatric atlas / spatial cohort to confirm progenitor-enriched composition.",
    ),
    dict(
        hypothesis_name="H_best4_centrality_residualized",
        result=(
            "BEST4 raw centrality is high (per first-pass LIANA), but after residualizing "
            "for cell-state abundance and the number of expressed ligands / receptors, "
            "BEST4 mean residual pagerank is **not** consistently greater than other "
            "epithelial cell states (best4_network_centrality_null_summary.csv: colon "
            "p_one_sided_greater≈0.98, ileum≈0.56, duodenum and jejunum each have only "
            "1 BEST4 state available so the MWU is unpowered). The dominant inferred "
            "top edges (S100A10–CFTR, TFF3–ACKR3) involve atypical ligand/receptor "
            "annotations that need orthogonal validation."),
        support_level="weakened_to_exploratory",
        main_supporting_table_or_figure="best4_network_centrality_null.csv, best4_network_centrality_null_summary.csv, liana_best4_edge_sanity_check.csv",
        caveats=(
            "LIANA centrality is an inferred summary that mixes (i) abundance, (ii) "
            "number of expressed ligands/receptors, and (iii) inferred edge strength. "
            "When (i) and (ii) are residualized, BEST4 no longer stands out as "
            "network-central in the segments we tested. Several top BEST4-incident edges "
            "(S100A10 as ligand; TFF3→ACKR3) involve biologically questionable LR "
            "annotations and should not be cited without orthogonal evidence."),
        recommended_paper_phrasing=(
            "After adjusting for cell-state abundance and the number of expressed "
            "ligands and receptors, BEST4 cells do not exhibit excess inferred "
            "cell-cell-communication centrality compared to other absorptive epithelial "
            "states; high centrality reported in unadjusted LIANA networks should be "
            "interpreted as a function of expressed gene count rather than a "
            "specific BEST4 signaling-hub identity."),
        recommended_next_validation=(
            "Re-run CCC with an orthogonal tool (CellChat, NicheNet) on the same matrices; "
            "validate the residualized centrality story with held-out gut atlases; "
            "biochemical validation of any specific top edge (e.g. S100A10→CFTR, "
            "TFF3→ACKR3) before citing it as a BEST4 signaling axis."),
    ),
    dict(
        hypothesis_name="H_mechanism_axis_assignment",
        result=("Mechanism modules summarized in diarrhea_mechanism_module_summary.csv. "
                f"Pediatric-vs-adult contrasts with q<0.1 (top 6 by FDR): {_mech_top_str}."),
        support_level=("strong" if (not mech_hits.empty and (mech_hits['q_fdr'] < 0.01).any())
                       else "moderate" if not mech_hits.empty else "exploratory"),
        main_supporting_table_or_figure="diarrhea_mechanism_module_summary.csv, diarrhea_mechanism_module_pediatric_vs_adult.csv, fig_2p5_mechanism_modules_heatmap",
        caveats="Modules are small and partly overlapping (CFTR appears in both ETEC_GCC and cAMP-CFTR axes; SLC9A3 in both Na-HCO3 and ORS axes). Jejunum and duodenum are underpowered. Direction of pediatric effect can flip across cell states even within the same module.",
        recommended_paper_phrasing=(
            "In pediatric samples, absorptive epithelial mechanism modules show a *cell-state-stratified* pattern: "
            "the GC-C / cGMP axis is **up** in BEST4 Enterocytes (small bowel) and **down** in colonic Crypt Progenitors; "
            "the cholera / cAMP-CFTR axis is **down** in mid- and lower-villus enterocytes and BEST4 Enterocytes (small bowel); "
            "the sodium-bicarbonate absorption axis and ORS axis are **up** in Crypt Top Colonocytes (colon) but **down** in Villus Tip and Mid Villus Enterocytes (small bowel). "
            "This points to *immature small-bowel absorptive maturation* rather than uniformly elevated secretory machinery as the dominant pediatric-specific signal."),
        recommended_next_validation="Pediatric-vs-adult validation cohort with stratified diarrhea etiology (ETEC vs Vibrio vs rotavirus); functional Ussing-chamber readouts of GC-C-driven Cl⁻ secretion and Na-glucose-driven absorption in pediatric vs adult villus-tip vs crypt-top epithelium.",
    ),
]
hypothesis_df = pd.DataFrame(hypothesis_rows)
hypothesis_df.to_csv(SECOND_PASS_DIR / "top_hypothesis_candidates_second_pass.csv", index=False)
print("Second-pass hypothesis ranking:")
print(hypothesis_df[["hypothesis_name", "support_level"]].to_string(index=False))

In [ ]:
# Output summary for the second pass
out_files = sorted(p.name for p in SECOND_PASS_DIR.glob("*"))
print(f"Second-pass produced {len(out_files)} files in {SECOND_PASS_DIR}:")
for f in out_files:
    print(f"  {f}")

# =============================================================================
# Third pass — clean, conservative age-related epithelial gene signature
# =============================================================================
#
# Uses the **full** HGCA all-lineages object so we can include enterocyte
# states that are missing from the epithelial-only file. Reports gene-level,
# cell-state-stratified age effects with strict sample-level coverage filters
# and leave-one-dataset-out sensitivity. Outputs go to the same second-pass
# directory.

In [ ]:
HGCA_ALL_PATH = "/Users/kylekimler/Projects/GCA/meta_datasets/integrated-objects/hgca_all_lineages_v1.h5ad"
P3_DIR = SECOND_PASS_DIR
print(f"Third-pass output dir: {P3_DIR}")

GENES_OF_INTEREST = [
    "DGAT1", "APOA1", "APOA4", "APOB", "MTTP", "FABP1", "FABP2", "FABP5",
    "SAR1B", "APOC3", "ALPI", "ALDOB", "SI",
    "SLC4A4", "SLC9A2", "SLC9A3", "SLC26A3", "SLC5A1",
    "CFTR", "GUCA2A", "GUCA2B", "GUCY2C",
    "BEST4", "OTOP2", "CA7", "CA2", "CA12",
]

ABSORPTIVE_PATTERNS = [
    "BEST4 Enterocytes", "BEST4 Colonocytes",
    "Villus Tip Enterocytes", "Mid Villus Enterocytes", "Lower Villus Enterocytes",
    "Enterocyte Progenitors", "Enterocytes",
    "Crypt Top Colonocytes", "Mid Crypt Colonocytes", "Lower Crypt Colonocytes",
    "Colonocyte Progenitors", "Colonocytes",
]

# Decade-bin midpoints used as numeric age proxy.
AGE_BIN_TO_NUMERIC = {
    "0-9": 5, "10-19": 15, "20-29": 25, "30-39": 35, "40-49": 45,
    "50-59": 55, "60-69": 65, "70-79": 75, "80-89": 85,
}

print("Loading HGCA all-lineages object (this is large)...")
adata_all = sc.read_h5ad(HGCA_ALL_PATH)
print(f"Loaded: {adata_all.shape}")
print(f"hgca_celltype_v1: {adata_all.obs['hgca_celltype_v1'].nunique()} unique types")

# Match absorptive labels case-insensitively
all_states_norm = {s.lower(): s for s in adata_all.obs["hgca_celltype_v1"].astype(str).unique()}
matched_states = []
for pat in ABSORPTIVE_PATTERNS:
    if pat.lower() in all_states_norm:
        matched_states.append(all_states_norm[pat.lower()])
matched_states = sorted(set(matched_states))
print(f"Matched absorptive states ({len(matched_states)}):")
for s in matched_states:
    print(f"  {s}")
pd.Series(matched_states, name="cell_state").to_csv(P3_DIR / "absorptive_states_matched.csv", index=False)

# Subset to absorptive cells only (in MAIN_TISSUES) BEFORE bringing into memory deeper
mask = (adata_all.obs["hgca_celltype_v1"].astype(str).isin(matched_states) &
        adata_all.obs["tissue_level_1"].isin(MAIN_TISSUES))
print(f"Absorptive cells in MAIN_TISSUES: {int(mask.sum()):,} / {adata_all.n_obs:,}")

# Now restrict to genes of interest
adata_all.var_names = adata_all.var["gene_symbol"].astype(str)
adata_all.var_names_make_unique()
present_genes = [g for g in GENES_OF_INTEREST if g in adata_all.var_names]
missing_genes = [g for g in GENES_OF_INTEREST if g not in adata_all.var_names]
print(f"Genes present: {len(present_genes)} / {len(GENES_OF_INTEREST)}")
if missing_genes:
    print(f"Missing genes (not in atlas var): {missing_genes}")
pd.Series(missing_genes, name="missing_gene").to_csv(P3_DIR / "age_signature_missing_genes.csv", index=False)

ad_p3 = adata_all[mask, present_genes].copy()
del adata_all  # free memory
print(f"ad_p3 shape: {ad_p3.shape}")

# Detect & ensure log1p-CPM expression for the gene matrix
X_test = ad_p3.X[:200, :].toarray() if sp.issparse(ad_p3.X) else np.asarray(ad_p3.X[:200, :])
if X_test.size and (X_test.max() > 50 and (X_test == X_test.astype(int)).all()):
    print("ad_p3.X looks like raw counts; normalizing in place to log1p CPM-equivalent.")
    sc.pp.normalize_total(ad_p3, target_sum=1e4)
    sc.pp.log1p(ad_p3)
else:
    print("ad_p3.X already looks normalized.")

# Age numeric + groups
age_str = ad_p3.obs["age_range"].astype(str)
ad_p3.obs["age_numeric"] = age_str.map(AGE_BIN_TO_NUMERIC)
ad_p3.obs["age_group"] = np.where(
    age_str == "0-9", "child_0_9",
    np.where(age_str == "10-19", "youth_10_19",
             np.where(age_str.isin(["20-29", "30-39", "40-49", "50-59", "60-69", "70-79", "80-89"]),
                       "adult", "unknown")))
ad_p3.obs["pediatric_0_19"] = ad_p3.obs["age_group"].isin(["child_0_9", "youth_10_19"])

print()
print("Cell counts by age group:")
print(ad_p3.obs["age_group"].value_counts().to_string())

In [ ]:
# Sample-level coverage table
coverage_keys = ["age_group", "tissue_level_1", "dataset_id", "hgca_celltype_v1"]
sample_unit_keys = ["sample_id", "dataset_id", "tissue_level_1",
                    "hgca_celltype_v1", "age_group", "age_numeric"]
if "donor_id" in ad_p3.obs.columns:
    sample_unit_keys.insert(1, "donor_id")

# Cells aggregated to sample-level summary entries (one row per sample x cellstate present)
cov_df = (ad_p3.obs.groupby(sample_unit_keys, observed=True)
          .size().reset_index(name="n_cells"))
cov_df.to_csv(P3_DIR / "age_signature_coverage_table.csv", index=False)
print(f"Sample x cellstate rows: {len(cov_df):,}")

# Wider sample-by-(age_group, tissue) coverage of distinct sample_ids and datasets
cov_wide_samples = (cov_df.groupby(["age_group", "tissue_level_1", "hgca_celltype_v1"], observed=True)
                    .agg(n_samples=("sample_id", "nunique"),
                         n_datasets=("dataset_id", "nunique"))
                    .reset_index())
cov_wide_samples.to_csv(P3_DIR / "age_signature_coverage_samples_per_group.csv", index=False)
print()
print("Coverage per (age_group, tissue, cell state): top sparse cells")
sparse = cov_wide_samples[cov_wide_samples["n_samples"] < 5].sort_values(
    ["tissue_level_1", "age_group", "hgca_celltype_v1"])
print(f"  {len(sparse)} (age_group, tissue, state) tuples have <5 samples — flagged as low-coverage.")
print(sparse.head(20).to_string(index=False))

In [ ]:
# Pseudobulk: sample x cell state x gene
group_id_p3 = ad_p3.obs[["sample_id", "hgca_celltype_v1"]].astype(str).agg("||".join, axis=1)
unique_groups_p3, group_codes_p3 = np.unique(group_id_p3.values, return_inverse=True)
n_groups_p3 = len(unique_groups_p3)
n_genes_p3 = len(present_genes)

X_p3 = ad_p3.X
mean_log_p3 = np.zeros((n_groups_p3, n_genes_p3), dtype=np.float32)
frac_pos_p3 = np.zeros((n_groups_p3, n_genes_p3), dtype=np.float32)
n_cells_p3 = np.zeros(n_groups_p3, dtype=np.int64)

if sp.issparse(X_p3):
    X_arr = X_p3.toarray().astype(np.float32)
else:
    X_arr = np.asarray(X_p3, dtype=np.float32)
X_pos = (X_arr > 0).astype(np.float32)

for k in range(n_groups_p3):
    rows = np.where(group_codes_p3 == k)[0]
    n_cells_p3[k] = len(rows)
    if rows.size:
        mean_log_p3[k] = X_arr[rows].mean(axis=0)
        frac_pos_p3[k] = X_pos[rows].mean(axis=0)

mean_p3_df = pd.DataFrame(mean_log_p3, index=unique_groups_p3, columns=present_genes)
frac_p3_df = pd.DataFrame(frac_pos_p3, index=unique_groups_p3, columns=present_genes)

# Attach metadata (one row per (sample, cellstate))
meta_keys = ["sample_id", "hgca_celltype_v1", "tissue_level_1", "dataset_id",
             "age_group", "age_numeric", "pediatric_0_19"]
if "donor_id" in ad_p3.obs.columns:
    meta_keys.insert(2, "donor_id")
for opt in ["assay", "sampled_site_condition",
            "sample_collection_method", "sample_preservation_method"]:
    if opt in ad_p3.obs.columns:
        meta_keys.append(opt)

meta_p3 = (ad_p3.obs.assign(_gid=group_id_p3)
           .drop_duplicates("_gid")
           .set_index("_gid")[meta_keys])
meta_p3 = meta_p3.reindex(unique_groups_p3)
meta_p3["n_cells"] = n_cells_p3

mean_p3_df = mean_p3_df.merge(meta_p3, left_index=True, right_index=True)
frac_p3_df = frac_p3_df.merge(meta_p3, left_index=True, right_index=True)

# Sample-level summary table in long form (one row per (sample, cellstate, gene))
long_rows = []
for gene in present_genes:
    long_rows.append(pd.DataFrame({
        "sample_id":          mean_p3_df["sample_id"].values,
        "donor_id":           mean_p3_df["donor_id"].values if "donor_id" in mean_p3_df.columns else np.nan,
        "dataset_id":         mean_p3_df["dataset_id"].values,
        "tissue_level_1":     mean_p3_df["tissue_level_1"].values,
        "hgca_celltype_v1":   mean_p3_df["hgca_celltype_v1"].values,
        "age_group":          mean_p3_df["age_group"].values,
        "age_numeric":        mean_p3_df["age_numeric"].values,
        "pediatric_0_19":     mean_p3_df["pediatric_0_19"].values,
        "n_cells":            mean_p3_df["n_cells"].values,
        "gene":               gene,
        "mean_log":           mean_p3_df[gene].values,
        "frac_detected":      frac_p3_df[gene].values,
    }))
long_df = pd.concat(long_rows, ignore_index=True)
long_df.to_csv(P3_DIR / "age_gene_summary_by_sample_cellstate.csv", index=False)
print(f"Long sample-level summary: {len(long_df):,} rows for {long_df['gene'].nunique()} genes "
      f"x {long_df['hgca_celltype_v1'].nunique()} cell states "
      f"x {long_df['sample_id'].nunique()} samples")

In [ ]:
N_CELLS_MIN_P3 = 20
N_SAMPLES_MIN_PER_GROUP = 5
N_DATASETS_MIN_PER_GROUP = 2


def _age_test_one(df, contrast, outcome, covariates):
    """Sample-level OLS for a single (gene, cell state).

    df: long_df subset with columns including outcome, dataset_id, tissue_level_1.
    contrast: 'pediatric_0_19', 'child_0_9', or 'continuous'.
    Returns dict of beta, p, n_*, coverage_flag, or None if uninstantiable.
    """
    sub = df.dropna(subset=[outcome]).copy()
    sub = sub[sub["n_cells"] >= N_CELLS_MIN_P3]
    if sub.empty:
        return None
    if contrast == "pediatric_0_19":
        sub = sub[sub["age_group"].isin(["child_0_9", "youth_10_19", "adult"])]
        sub["x"] = sub["age_group"].isin(["child_0_9", "youth_10_19"]).astype(int)
    elif contrast == "child_0_9":
        sub = sub[sub["age_group"].isin(["child_0_9", "adult"])]
        sub["x"] = (sub["age_group"] == "child_0_9").astype(int)
    elif contrast == "continuous":
        sub = sub.dropna(subset=["age_numeric"])
        sub["x"] = sub["age_numeric"].astype(float)
    else:
        return None

    if contrast in ("pediatric_0_19", "child_0_9"):
        n_ped = int(sub.loc[sub["x"] == 1, "sample_id"].nunique())
        n_adu = int(sub.loc[sub["x"] == 0, "sample_id"].nunique())
        n_ds_ped = int(sub.loc[sub["x"] == 1, "dataset_id"].nunique())
        n_ds_adu = int(sub.loc[sub["x"] == 0, "dataset_id"].nunique())
        cov_flag = ("ok" if (n_ped >= N_SAMPLES_MIN_PER_GROUP and
                              n_adu >= N_SAMPLES_MIN_PER_GROUP and
                              n_ds_ped >= N_DATASETS_MIN_PER_GROUP and
                              n_ds_adu >= N_DATASETS_MIN_PER_GROUP)
                    else "insufficient")
    else:
        n_ped = int(sub["sample_id"].nunique())
        n_adu = 0
        n_ds_ped = int(sub["dataset_id"].nunique())
        n_ds_adu = 0
        cov_flag = ("ok" if (n_ped >= 10 and n_ds_ped >= N_DATASETS_MIN_PER_GROUP
                              and sub["x"].nunique() >= 3)
                    else "insufficient")

    if cov_flag == "insufficient":
        return dict(coverage_flag=cov_flag, beta=np.nan, se=np.nan, p_value=np.nan,
                    n_samples_pediatric=n_ped, n_samples_adult=n_adu,
                    n_datasets_pediatric=n_ds_ped, n_datasets_adult=n_ds_adu,
                    n_donors=int(sub["donor_id"].nunique()) if "donor_id" in sub.columns else np.nan,
                    n_obs=len(sub))

    usable = []
    for term in covariates:
        if term in sub.columns and sub[term].nunique() >= 2:
            usable.append(f"C({term})")
    formula = f"{outcome} ~ x"
    if usable:
        formula += " + " + " + ".join(usable)
    try:
        res = smf.ols(formula, data=sub).fit(cov_type="HC3")
    except Exception:
        return dict(coverage_flag="model_failed", beta=np.nan, se=np.nan, p_value=np.nan,
                    n_samples_pediatric=n_ped, n_samples_adult=n_adu,
                    n_datasets_pediatric=n_ds_ped, n_datasets_adult=n_ds_adu,
                    n_donors=int(sub["donor_id"].nunique()) if "donor_id" in sub.columns else np.nan,
                    n_obs=len(sub))
    return dict(
        coverage_flag=cov_flag,
        beta=float(res.params.get("x", np.nan)),
        se=float(res.bse.get("x", np.nan)),
        p_value=float(res.pvalues.get("x", np.nan)),
        n_samples_pediatric=n_ped,
        n_samples_adult=n_adu,
        n_datasets_pediatric=n_ds_ped,
        n_datasets_adult=n_ds_adu,
        n_donors=int(sub["donor_id"].nunique()) if "donor_id" in sub.columns else np.nan,
        n_obs=len(sub),
    )


def _run_all_tests(long_df, contrast, outcome="mean_log",
                   covariates=("tissue_level_1", "dataset_id")):
    rows = []
    for (gene, ct), g in long_df.groupby(["gene", "hgca_celltype_v1"], observed=True):
        r = _age_test_one(g, contrast, outcome, list(covariates))
        if r is None:
            continue
        r.update(dict(gene=gene, cell_state=ct, contrast=contrast, outcome=outcome))
        rows.append(r)
    out = pd.DataFrame(rows)
    if out.empty:
        return out
    out["q_fdr"] = np.nan
    ok = out["coverage_flag"] == "ok"
    if ok.any():
        ps = out.loc[ok, "p_value"].fillna(1.0).values
        _, q, _, _ = multipletests(ps, method="fdr_bh")
        out.loc[ok, "q_fdr"] = q
    return out.sort_values(["coverage_flag", "q_fdr", "p_value"]).reset_index(drop=True)


tests_ped = _run_all_tests(long_df, "pediatric_0_19", "mean_log")
tests_ped_frac = _run_all_tests(long_df, "pediatric_0_19", "frac_detected")
tests_child = _run_all_tests(long_df, "child_0_9", "mean_log")
tests_cont = _run_all_tests(long_df, "continuous", "mean_log")

tests_ped.to_csv(P3_DIR / "age_gene_tests_pediatric_0_19_vs_adult.csv", index=False)
tests_ped_frac.to_csv(P3_DIR / "age_gene_tests_pediatric_0_19_vs_adult_frac.csv", index=False)
if not tests_child.empty:
    tests_child.to_csv(P3_DIR / "age_gene_tests_child_0_9_vs_adult.csv", index=False)
if not tests_cont.empty:
    tests_cont.to_csv(P3_DIR / "age_gene_tests_continuous_age.csv", index=False)

print(f"pediatric_0_19 vs adult tests: {len(tests_ped)} rows")
print(f"  ok coverage: {(tests_ped['coverage_flag']=='ok').sum()}")
print(f"  q < 0.1: {(tests_ped['q_fdr']<0.1).sum()}, q < 0.05: {(tests_ped['q_fdr']<0.05).sum()}")
print()
print("Top 15 ok-coverage hits (mean_log, pediatric_0_19 vs adult):")
ok_ped = tests_ped[tests_ped["coverage_flag"] == "ok"].sort_values("q_fdr").head(15)
print(ok_ped[["gene", "cell_state", "beta", "se", "p_value", "q_fdr",
              "n_samples_pediatric", "n_samples_adult",
              "n_datasets_pediatric", "n_datasets_adult"]].to_string(index=False))

In [ ]:
PRIORITY_LODO = [
    ("DGAT1",   ["BEST4 Enterocytes", "Villus Tip Enterocytes"]),
    ("SLC4A4",  ["Mid Crypt Colonocytes", "Lower Crypt Colonocytes", "Colonocyte Progenitors"]),
    ("SLC9A2",  ["BEST4 Colonocytes", "Colonocyte Progenitors"]),
    ("CFTR",    ["Colonocyte Progenitors"]),
    ("GUCA2B",  ["Villus Tip Enterocytes"]),
]

# Also LODO every pair that came out at q<0.1 from this atlas, so the final
# table reflects the actual hits.
denovo_top = (tests_ped[(tests_ped["coverage_flag"] == "ok") &
                         (tests_ped["q_fdr"] < 0.1)]
              .sort_values("q_fdr"))
denovo_lodo_pairs = list(zip(denovo_top["gene"].tolist(),
                              denovo_top["cell_state"].tolist()))
PRIORITY_LODO_FLAT = []
for gene, cts in PRIORITY_LODO:
    for ct in cts:
        PRIORITY_LODO_FLAT.append((gene, ct))
for pair in denovo_lodo_pairs:
    if pair not in PRIORITY_LODO_FLAT:
        PRIORITY_LODO_FLAT.append(pair)


def _lodo_age_test(long_df_sub, gene, ct, contrast, outcome="mean_log",
                   covariates=("tissue_level_1", "dataset_id")):
    """Returns dict with full-fit beta+q and per-LODO beta/q/sign-consistency."""
    g = long_df_sub[(long_df_sub["gene"] == gene) &
                    (long_df_sub["hgca_celltype_v1"] == ct)]
    full_res = _age_test_one(g, contrast, outcome, list(covariates))
    if full_res is None or full_res.get("coverage_flag") != "ok":
        return dict(gene=gene, cell_state=ct, contrast=contrast,
                    full_beta=np.nan, full_p=np.nan, full_q=np.nan,
                    n_datasets=0, lodo_n=0,
                    lodo_betas_min=np.nan, lodo_betas_max=np.nan,
                    lodo_sign_consistent=False, lodo_q_below_0_1_in=0,
                    coverage_flag=(full_res or {}).get("coverage_flag", "insufficient"))
    full_beta = full_res["beta"]
    full_p = full_res["p_value"]
    sub = g.dropna(subset=[outcome]).copy()
    sub = sub[sub["n_cells"] >= N_CELLS_MIN_P3]
    datasets = sub["dataset_id"].unique()
    lodo_betas = []
    lodo_qs = []
    sign_consistent_count = 0
    for ds in datasets:
        kept = sub[sub["dataset_id"] != ds]
        r = _age_test_one(kept, contrast, outcome, list(covariates))
        if r is None or pd.isna(r.get("beta")) or r.get("coverage_flag") != "ok":
            continue
        lodo_betas.append(r["beta"])
        lodo_qs.append(r["p_value"])  # we'll use raw p; q-style threshold here is informal
        if (np.sign(r["beta"]) == np.sign(full_beta)) and not (full_beta == 0):
            sign_consistent_count += 1
    if not lodo_betas:
        return dict(gene=gene, cell_state=ct, contrast=contrast,
                    full_beta=full_beta, full_p=full_p, full_q=np.nan,
                    n_datasets=len(datasets), lodo_n=0,
                    lodo_betas_min=np.nan, lodo_betas_max=np.nan,
                    lodo_sign_consistent=False, lodo_q_below_0_1_in=0,
                    coverage_flag="ok")
    lodo_arr = np.array(lodo_betas)
    return dict(
        gene=gene, cell_state=ct, contrast=contrast,
        full_beta=full_beta, full_p=full_p, full_q=np.nan,
        n_datasets=int(len(datasets)),
        lodo_n=int(len(lodo_betas)),
        lodo_betas_min=float(lodo_arr.min()),
        lodo_betas_max=float(lodo_arr.max()),
        lodo_sign_consistent=(sign_consistent_count == len(lodo_betas)),
        lodo_p_below_0_1_in=int(sum(p < 0.1 for p in lodo_qs)),
        lodo_p_below_0_05_in=int(sum(p < 0.05 for p in lodo_qs)),
        coverage_flag="ok",
    )


lodo_rows = []
for gene, ct in PRIORITY_LODO_FLAT:
    for contrast in ["pediatric_0_19", "child_0_9"]:
        r = _lodo_age_test(long_df, gene, ct, contrast, "mean_log")
        r["outcome"] = "mean_log"
        lodo_rows.append(r)

# Attach the q value from the main test to each LODO row
qmap_ped = (tests_ped.set_index(["gene", "cell_state"])["q_fdr"]
            .to_dict())
qmap_child = (tests_child.set_index(["gene", "cell_state"])["q_fdr"].to_dict()
              if not tests_child.empty else {})
for r in lodo_rows:
    key = (r["gene"], r["cell_state"])
    if r["contrast"] == "pediatric_0_19":
        r["full_q"] = qmap_ped.get(key, np.nan)
    elif r["contrast"] == "child_0_9":
        r["full_q"] = qmap_child.get(key, np.nan)


# Robustness label
def _robust_label(r):
    if r.get("coverage_flag") != "ok" or pd.isna(r.get("full_beta")):
        return "insufficient"
    if r["lodo_n"] < 3:
        return "fragile"
    sign_ok = bool(r["lodo_sign_consistent"])
    p_below_0_1_share = r["lodo_p_below_0_1_in"] / max(r["lodo_n"], 1)
    if sign_ok and p_below_0_1_share >= 0.7:
        return "robust"
    if sign_ok or p_below_0_1_share >= 0.5:
        return "mixed"
    return "fragile"


lodo_df = pd.DataFrame(lodo_rows)
if not lodo_df.empty:
    lodo_df["robustness"] = lodo_df.apply(_robust_label, axis=1)
    lodo_df.to_csv(P3_DIR / "age_signature_lodo_sensitivity.csv", index=False)
    print("LODO sensitivity for priority hits:")
    print(lodo_df[["gene", "cell_state", "contrast", "full_beta", "full_q",
                   "lodo_n", "lodo_betas_min", "lodo_betas_max",
                   "lodo_sign_consistent", "lodo_p_below_0_1_in",
                   "lodo_p_below_0_05_in", "robustness"]].to_string(index=False))

In [ ]:
BEST4_STATES = [s for s in matched_states if s.lower().startswith("best4")]
print("BEST4 states identified:", BEST4_STATES)


# Sample-level fraction of BEST4 cells among absorptive-epithelial cells
abs_per_sample = (ad_p3.obs.groupby(["sample_id", "tissue_level_1", "dataset_id",
                                      "age_group", "age_numeric"], observed=True)
                  .size().reset_index(name="n_absorptive_cells"))
best4_per_sample = (
    ad_p3.obs[ad_p3.obs["hgca_celltype_v1"].isin(BEST4_STATES)]
    .groupby(["sample_id", "tissue_level_1", "dataset_id",
              "age_group", "age_numeric", "hgca_celltype_v1"], observed=True)
    .size().reset_index(name="n_best4_cells")
)
best4_abund = best4_per_sample.merge(abs_per_sample,
                                      on=["sample_id", "tissue_level_1", "dataset_id",
                                          "age_group", "age_numeric"],
                                      how="left")
best4_abund["frac_best4_in_absorptive"] = best4_abund["n_best4_cells"] / best4_abund["n_absorptive_cells"]
best4_abund["pediatric_0_19"] = best4_abund["age_group"].isin(["child_0_9", "youth_10_19"]).astype(int)
best4_abund.to_csv(P3_DIR / "age_best4_abundance_summary.csv", index=False)
print(f"BEST4 abundance rows: {len(best4_abund)}")
print()


abund_test_rows = []
for state, g in best4_abund.groupby("hgca_celltype_v1", observed=True):
    g = g.dropna(subset=["frac_best4_in_absorptive"])
    if g.empty:
        continue
    n_ped = int(g.loc[g["pediatric_0_19"] == 1, "sample_id"].nunique())
    n_adu = int(g.loc[g["pediatric_0_19"] == 0, "sample_id"].nunique())
    n_ds_ped = int(g.loc[g["pediatric_0_19"] == 1, "dataset_id"].nunique())
    n_ds_adu = int(g.loc[g["pediatric_0_19"] == 0, "dataset_id"].nunique())
    cov_flag = ("ok" if (n_ped >= N_SAMPLES_MIN_PER_GROUP and
                          n_adu >= N_SAMPLES_MIN_PER_GROUP and
                          n_ds_ped >= N_DATASETS_MIN_PER_GROUP and
                          n_ds_adu >= N_DATASETS_MIN_PER_GROUP)
                else "insufficient")
    if cov_flag != "ok":
        abund_test_rows.append(dict(cell_state=state, scope="all_segments",
                                     coverage_flag=cov_flag,
                                     n_samples_pediatric=n_ped,
                                     n_samples_adult=n_adu,
                                     n_datasets_pediatric=n_ds_ped,
                                     n_datasets_adult=n_ds_adu,
                                     beta=np.nan, se=np.nan, p_value=np.nan))
        continue
    formula = "frac_best4_in_absorptive ~ pediatric_0_19"
    if g["tissue_level_1"].nunique() > 1:
        formula += " + C(tissue_level_1)"
    if g["dataset_id"].nunique() > 1:
        formula += " + C(dataset_id)"
    try:
        res = smf.ols(formula, data=g).fit(cov_type="HC3")
        abund_test_rows.append(dict(
            cell_state=state, scope="all_segments", coverage_flag="ok",
            n_samples_pediatric=n_ped, n_samples_adult=n_adu,
            n_datasets_pediatric=n_ds_ped, n_datasets_adult=n_ds_adu,
            beta=float(res.params.get("pediatric_0_19", np.nan)),
            se=float(res.bse.get("pediatric_0_19", np.nan)),
            p_value=float(res.pvalues.get("pediatric_0_19", np.nan)),
        ))
    except Exception:
        abund_test_rows.append(dict(cell_state=state, scope="all_segments",
                                     coverage_flag="model_failed",
                                     n_samples_pediatric=n_ped,
                                     n_samples_adult=n_adu,
                                     n_datasets_pediatric=n_ds_ped,
                                     n_datasets_adult=n_ds_adu,
                                     beta=np.nan, se=np.nan, p_value=np.nan))

abund_tests = pd.DataFrame(abund_test_rows)
abund_tests.to_csv(P3_DIR / "age_best4_abundance_tests.csv", index=False)
print("BEST4 abundance pediatric_0_19 vs adult tests:")
print(abund_tests.to_string(index=False))

In [ ]:
# --- Plot 1: coverage heatmap ------------------------------------------------
age_group_order = ["child_0_9", "youth_10_19", "adult"]
cov_pivot_samples = (cov_wide_samples
                     .pivot_table(index=["hgca_celltype_v1"],
                                  columns=["age_group", "tissue_level_1"],
                                  values="n_samples", aggfunc="sum")
                     .reindex(index=matched_states))

fig, ax = plt.subplots(figsize=mm(180, 90))
arr = cov_pivot_samples.fillna(0).astype(int).values
im = ax.imshow(arr, cmap="viridis", aspect="auto")
ax.set_yticks(range(len(cov_pivot_samples.index)))
ax.set_yticklabels(cov_pivot_samples.index, fontsize=4.5)
xtl = ["\n".join(map(str, c)) for c in cov_pivot_samples.columns]
ax.set_xticks(range(len(xtl)))
ax.set_xticklabels(xtl, fontsize=4, rotation=90)
for i in range(arr.shape[0]):
    for j in range(arr.shape[1]):
        v = int(arr[i, j])
        if v > 0:
            ax.text(j, i, str(v),
                    ha="center", va="center", fontsize=3.5,
                    color="white" if arr.max() and v / max(arr.max(), 1) < 0.5 else "black")
ax.set_title("Sample coverage of absorptive states (n_samples per age_group x tissue_level_1)",
              fontsize=6)
cb = fig.colorbar(im, ax=ax, fraction=0.04, pad=0.02)
cb.ax.tick_params(labelsize=4.5)
fig.tight_layout()
save_figure(fig, P3_DIR / "fig_3p_coverage_age_x_tissue", 180, 90)


# --- Plot 2: BEST4 abundance by age group ------------------------------------
fig, axes = plt.subplots(1, len(BEST4_STATES) if BEST4_STATES else 1,
                         figsize=mm(180, 75), squeeze=False)
for ax, state in zip(axes[0], BEST4_STATES):
    g = best4_abund[best4_abund["hgca_celltype_v1"] == state].copy()
    if g.empty:
        ax.set_visible(False)
        continue
    g["age_group_short"] = g["age_group"].map({"child_0_9": "0-9",
                                                 "youth_10_19": "10-19",
                                                 "adult": "adult"})
    age_order_local = [v for v in ["0-9", "10-19", "adult"]
                       if v in g["age_group_short"].unique()]
    sns.boxplot(data=g, x="age_group_short", y="frac_best4_in_absorptive",
                order=age_order_local, ax=ax,
                width=0.5, fliersize=0,
                boxprops=dict(facecolor="white", edgecolor="black", linewidth=0.5),
                medianprops=dict(color="black", linewidth=0.6),
                whiskerprops=dict(color="black", linewidth=0.4),
                capprops=dict(color="black", linewidth=0.4))
    for ag, marker_x in zip(age_order_local, range(len(age_order_local))):
        sub = g[g["age_group_short"] == ag]
        for tissue in sub["tissue_level_1"].unique():
            tt = sub[sub["tissue_level_1"] == tissue]
            jitter = (np.random.RandomState(0).rand(len(tt)) - 0.5) * 0.25
            ax.scatter(np.full(len(tt), marker_x) + jitter,
                       tt["frac_best4_in_absorptive"],
                       s=6, alpha=0.85,
                       color=SEGMENT_COLORS.get(tissue, WONG["midgrey"]),
                       edgecolor="black", linewidths=0.2,
                       label=tissue if marker_x == 0 else None)
    test = abund_tests[abund_tests["cell_state"] == state]
    if not test.empty and pd.notna(test.iloc[0].get("p_value")):
        ax.text(0.04, 0.95,
                f"beta={test.iloc[0]['beta']:+.3f}, p={test.iloc[0]['p_value']:.2g}",
                transform=ax.transAxes, va="top", ha="left", fontsize=5)
    ax.set_xlabel("age group")
    ax.set_ylabel("fraction of absorptive cells")
    ax.set_title(state, fontsize=6)
    if ax is axes[0, 0]:
        ax.legend(fontsize=4, frameon=False, loc="upper right")
    style_ax(ax)
fig.suptitle("Sample-level BEST4 abundance vs age group", fontsize=7)
fig.tight_layout(rect=[0, 0, 1, 0.95])
save_figure(fig, P3_DIR / "fig_3p_best4_abundance_by_age", 180, 75)

In [ ]:
# --- Plot 3: age-effect heatmap (gene x cell state, beta for pediatric_0_19) ----
PRIORITY_GENES_PLOT = [
    "DGAT1", "APOA1", "APOA4", "MTTP", "FABP1", "FABP2", "FABP5",
    "ALPI", "ALDOB", "SI",
    "SLC4A4", "SLC9A2", "SLC9A3", "SLC26A3", "SLC5A1",
    "CFTR", "GUCA2B", "GUCY2C", "BEST4", "OTOP2", "CA7",
]
plot_genes = [g for g in PRIORITY_GENES_PLOT if g in tests_ped["gene"].unique()]

heat_beta = (tests_ped[(tests_ped["coverage_flag"] == "ok") &
                        (tests_ped["gene"].isin(plot_genes)) &
                        (tests_ped["cell_state"].isin(matched_states))]
             .pivot_table(index="gene", columns="cell_state", values="beta"))
heat_q = (tests_ped[(tests_ped["coverage_flag"] == "ok") &
                     (tests_ped["gene"].isin(plot_genes)) &
                     (tests_ped["cell_state"].isin(matched_states))]
          .pivot_table(index="gene", columns="cell_state", values="q_fdr"))
heat_beta = heat_beta.reindex(index=plot_genes, columns=matched_states)
heat_q = heat_q.reindex_like(heat_beta)

fig, ax = plt.subplots(figsize=mm(180, 110))
arr = heat_beta.values
vmax = float(np.nanmax(np.abs(arr))) if np.isfinite(np.nanmax(np.abs(arr))) else 1.0
im = ax.imshow(arr, cmap="RdBu_r", vmin=-vmax, vmax=vmax, aspect="auto")
ax.set_yticks(range(len(heat_beta.index)))
ax.set_yticklabels(heat_beta.index, fontsize=5)
ax.set_xticks(range(len(heat_beta.columns)))
ax.set_xticklabels(heat_beta.columns, fontsize=5, rotation=45, ha="right")
for i in range(arr.shape[0]):
    for j in range(arr.shape[1]):
        v = arr[i, j]
        q = heat_q.values[i, j]
        if pd.notna(q) and q < 0.05:
            ax.text(j, i, "**", ha="center", va="center", fontsize=5,
                    color="black" if abs(v) < vmax * 0.5 else "white")
        elif pd.notna(q) and q < 0.1:
            ax.text(j, i, "*", ha="center", va="center", fontsize=5,
                    color="black" if abs(v) < vmax * 0.5 else "white")
ax.set_title("Pediatric (0-19) vs adult: beta on sample-level mean log expression\n"
              "(*=q<0.1, **=q<0.05; only ok-coverage cells shown)", fontsize=6)
cb = fig.colorbar(im, ax=ax, fraction=0.025, pad=0.02)
cb.ax.tick_params(labelsize=5)
cb.set_label("beta (pediatric_0_19 - adult)", fontsize=5)
fig.tight_layout()
save_figure(fig, P3_DIR / "fig_3p_age_effect_heatmap", 180, 110)

In [ ]:
# --- Plot 4: focused sample-level plots for top single-gene findings ---------
FOCUS_PAIRS = [
    ("DGAT1", "BEST4 Enterocytes"),
    ("DGAT1", "Villus Tip Enterocytes"),
    ("SLC4A4", "Colonocyte Progenitors"),
    ("SLC4A4", "Mid Crypt Colonocytes"),
    ("SLC9A2", "BEST4 Colonocytes"),
    ("GUCA2B", "Villus Tip Enterocytes"),
]

ncol = 3
nrow = int(np.ceil(len(FOCUS_PAIRS) / ncol))
fig, axes = plt.subplots(nrow, ncol, figsize=mm(180, 50 * nrow), squeeze=False)
for ax, (gene, ct) in zip(axes.ravel(), FOCUS_PAIRS):
    g = long_df[(long_df["gene"] == gene) &
                (long_df["hgca_celltype_v1"] == ct) &
                (long_df["n_cells"] >= N_CELLS_MIN_P3) &
                (long_df["age_group"].isin(["child_0_9", "youth_10_19", "adult"]))].copy()
    if g.empty:
        ax.set_visible(False); continue
    g["age_group_short"] = g["age_group"].map({"child_0_9": "0-9",
                                                 "youth_10_19": "10-19",
                                                 "adult": "adult"})
    age_order_local = [v for v in ["0-9", "10-19", "adult"]
                       if v in g["age_group_short"].unique()]
    sns.boxplot(data=g, x="age_group_short", y="mean_log",
                order=age_order_local, ax=ax,
                width=0.5, fliersize=0,
                boxprops=dict(facecolor="white", edgecolor="black", linewidth=0.5),
                medianprops=dict(color="black", linewidth=0.6),
                whiskerprops=dict(color="black", linewidth=0.4),
                capprops=dict(color="black", linewidth=0.4))
    for marker_x, ag in enumerate(age_order_local):
        sub = g[g["age_group_short"] == ag]
        for tissue in sub["tissue_level_1"].unique():
            tt = sub[sub["tissue_level_1"] == tissue]
            jitter = (np.random.RandomState(0).rand(len(tt)) - 0.5) * 0.25
            ax.scatter(np.full(len(tt), marker_x) + jitter,
                       tt["mean_log"],
                       s=6, alpha=0.85,
                       color=SEGMENT_COLORS.get(tissue, WONG["midgrey"]),
                       edgecolor="black", linewidths=0.2,
                       label=tissue if marker_x == 0 else None)
    test = tests_ped[(tests_ped["gene"] == gene) &
                      (tests_ped["cell_state"] == ct)]
    if not test.empty and test.iloc[0]["coverage_flag"] == "ok":
        r = test.iloc[0]
        ax.text(0.04, 0.95,
                f"beta={r['beta']:+.3f}, q={r['q_fdr']:.2g}\nn_ped={int(r['n_samples_pediatric'])}, n_adu={int(r['n_samples_adult'])}",
                transform=ax.transAxes, va="top", ha="left", fontsize=5)
    ax.set_xlabel("age group")
    ax.set_ylabel(f"{gene} mean log1p")
    ax.set_title(f"{gene} in {ct}", fontsize=6)
    if ax is axes[0, 0]:
        ax.legend(fontsize=4, frameon=False, loc="upper right")
    style_ax(ax)
for k in range(len(FOCUS_PAIRS), nrow * ncol):
    axes.ravel()[k].set_visible(False)
fig.tight_layout()
save_figure(fig, P3_DIR / "fig_3p_focused_top_findings", 180, 50 * nrow)

In [ ]:
# --- Final clean summary table -------------------------------------------------
# Combine the user-specified priority pairs with the strongest data-driven hits
# from this analysis (q<0.1 ok-coverage in pediatric_0_19 vs adult).
PRIORITY_PAIRS_FROM_PREVIOUS_PASS = [
    ("DGAT1",  "BEST4 Enterocytes",       "DGAT1 up in pediatric BEST4 Enterocytes (small bowel)",
     "DGAT1 sample-level expression in pediatric (<20y) small-bowel BEST4 Enterocytes."),
    ("DGAT1",  "Villus Tip Enterocytes",  "DGAT1 up in pediatric Villus Tip Enterocytes (small bowel)",
     "DGAT1 sample-level expression in pediatric (<20y) small-bowel Villus Tip Enterocytes."),
    ("SLC4A4", "Colonocyte Progenitors",  "SLC4A4 down in pediatric Colonocyte Progenitors",
     "SLC4A4 sample-level expression in pediatric (<20y) Colonocyte Progenitors."),
    ("SLC4A4", "Mid Crypt Colonocytes",   "SLC4A4 down in pediatric Mid Crypt Colonocytes",
     "SLC4A4 sample-level expression in pediatric (<20y) Mid Crypt Colonocytes."),
    ("SLC4A4", "Lower Crypt Colonocytes", "SLC4A4 down in pediatric Lower Crypt Colonocytes",
     "SLC4A4 sample-level expression in pediatric (<20y) Lower Crypt Colonocytes."),
    ("SLC9A2", "BEST4 Colonocytes",       "SLC9A2 down in pediatric BEST4 Colonocytes",
     "SLC9A2 sample-level expression in pediatric (<20y) BEST4 Colonocytes."),
    ("SLC9A2", "Colonocyte Progenitors",  "SLC9A2 down in pediatric Colonocyte Progenitors",
     "SLC9A2 sample-level expression in pediatric (<20y) Colonocyte Progenitors."),
    ("CFTR",   "Colonocyte Progenitors",  "CFTR down in pediatric Colonocyte Progenitors",
     "CFTR sample-level expression in pediatric (<20y) Colonocyte Progenitors."),
    ("GUCA2B", "Villus Tip Enterocytes",  "GUCA2B up in pediatric Villus Tip Enterocytes",
     "GUCA2B sample-level expression in pediatric (<20y) Villus Tip Enterocytes."),
]

# Top de-novo hits surfaced in this analysis (ok-coverage q<0.1 in pediatric_0_19)
denovo_hits_top = (tests_ped[(tests_ped["coverage_flag"] == "ok") &
                              (tests_ped["q_fdr"] < 0.1)]
                   .sort_values("q_fdr"))
DENOVO_PAIRS = []
for _, r in denovo_hits_top.iterrows():
    label_dir = "up" if r["beta"] > 0 else "down"
    DENOVO_PAIRS.append((
        r["gene"], r["cell_state"],
        f"{r['gene']} {label_dir} in pediatric {r['cell_state']}",
        f"{r['gene']} sample-level expression in pediatric (<20y) {r['cell_state']}.",
    ))
# Deduplicate
seen = set()
FINAL_FINDINGS = []
for entry in DENOVO_PAIRS + PRIORITY_PAIRS_FROM_PREVIOUS_PASS:
    key = (entry[0], entry[1])
    if key in seen:
        continue
    seen.add(key)
    FINAL_FINDINGS.append(entry)


def _support_level_from_lodo(robust_label, full_q):
    """Conservative support label.

    Rules:
    - q < 0.05 + LODO robust  -> strong
    - q < 0.05 + LODO mixed   -> moderate
    - q < 0.1  + LODO robust  -> moderate
    - q < 0.1  + LODO mixed   -> exploratory
    - q < 0.25                -> exploratory (only if LODO not actively bad)
    - else                    -> unsupported
    """
    if pd.isna(full_q):
        return "unsupported"
    if full_q < 0.05 and robust_label == "robust":
        return "strong"
    if full_q < 0.05:
        return "moderate"
    if full_q < 0.1 and robust_label == "robust":
        return "moderate"
    if full_q < 0.1:
        return "exploratory"
    if full_q < 0.25:
        return "exploratory"
    return "unsupported"


final_rows = []
for (gene, ct, finding, phrase_template) in FINAL_FINDINGS:
    test_row = tests_ped[(tests_ped["gene"] == gene) &
                          (tests_ped["cell_state"] == ct)]
    if test_row.empty:
        continue
    test_row = test_row.iloc[0]
    lodo_row = lodo_df[(lodo_df["gene"] == gene) &
                       (lodo_df["cell_state"] == ct) &
                       (lodo_df["contrast"] == "pediatric_0_19")]
    if lodo_row.empty:
        robustness = "not_computed"
    else:
        robustness = lodo_row.iloc[0]["robustness"]
    support = _support_level_from_lodo(robustness, test_row.get("q_fdr", np.nan))
    direction = "up" if (pd.notna(test_row.get("beta")) and test_row.get("beta", 0) > 0) else "down"
    caveat_bits = []
    if test_row.get("coverage_flag") != "ok":
        caveat_bits.append(f"coverage_flag={test_row.get('coverage_flag')}")
    if pd.isna(test_row.get("q_fdr")):
        caveat_bits.append("q not computable")
    elif test_row.get("q_fdr", 0) >= 0.99:
        caveat_bits.append("q ≈ 1: pediatric coefficient likely absorbed by dataset "
                            "fixed effects (dataset confounding); LODO subsets may "
                            "still show a directional signal")
    if robustness in {"mixed", "fragile"}:
        caveat_bits.append(f"LODO sensitivity {robustness}; treat as exploratory")
    elif robustness == "not_computed":
        caveat_bits.append("LODO not computed for this pair")
    if int(test_row.get("n_samples_pediatric", 0)) < 8:
        caveat_bits.append("pediatric sample count low (<8)")
    if int(test_row.get("n_datasets_pediatric", 0)) < 3:
        caveat_bits.append("only a small number of pediatric datasets contributed")
    final_rows.append(dict(
        finding=finding,
        gene=gene,
        cell_state=ct,
        direction=direction,
        beta=float(test_row.get("beta", np.nan))
              if pd.notna(test_row.get("beta")) else np.nan,
        q_value=float(test_row.get("q_fdr", np.nan))
                 if pd.notna(test_row.get("q_fdr")) else np.nan,
        n_samples_pediatric=int(test_row.get("n_samples_pediatric", 0)),
        n_samples_adult=int(test_row.get("n_samples_adult", 0)),
        n_datasets_pediatric=int(test_row.get("n_datasets_pediatric", 0)),
        n_datasets_adult=int(test_row.get("n_datasets_adult", 0)),
        lodo_sign_consistency=robustness,
        support_level=support,
        caveat="; ".join(caveat_bits) if caveat_bits else "",
        suggested_paper_phrase=phrase_template,
    ))


# Add the explicit BEST4-abundance negative result row.
# We test the user-asked direction ("elevated in pediatric") and ALSO note when
# the data show a significant *opposite* effect.
for state in BEST4_STATES:
    abund_row = abund_tests[abund_tests["cell_state"] == state]
    if abund_row.empty:
        continue
    abund_row = abund_row.iloc[0]
    if abund_row.get("coverage_flag") != "ok" or pd.isna(abund_row.get("p_value")):
        support = "unsupported"
        caveat = "Insufficient coverage to test pediatric vs adult abundance."
        phrase = (f"Sample-level abundance of {state} as a fraction of absorptive "
                  "epithelium could not be evaluated for pediatric vs adult under "
                  "strict coverage rules.")
    else:
        b = abund_row["beta"]; p = abund_row["p_value"]
        # The asked-for hypothesis is "increased in pediatric"
        if p < 0.05 and b > 0:
            support = "moderate"
            caveat = ""
            phrase = (f"Sample-level abundance of {state} as a fraction of absorptive "
                      f"epithelium is elevated in pediatric (<20y) relative to adult "
                      f"(β = {b:+.3f}, p ≈ {p:.2g}) after adjusting for tissue and dataset.")
        elif p < 0.1 and b > 0:
            support = "exploratory"
            caveat = "Borderline; would not survive multiple-testing correction."
            phrase = (f"Sample-level abundance of {state} shows a borderline "
                      f"elevation in pediatric (β = {b:+.3f}, p ≈ {p:.2g}).")
        else:
            support = "unsupported"
            opposite_note = ""
            if p < 0.05 and b < 0:
                opposite_note = (" The data instead show a *significant decrease* in "
                                  "pediatric abundance for this cell state — i.e. the "
                                  "elevation hypothesis is actively rejected, not just "
                                  "non-significant.")
            caveat = ("This is a negative-result row: pediatric does NOT show "
                      "elevated abundance for this BEST4 state after adjusting for "
                      f"tissue and dataset (β = {b:+.3f}, p ≈ {p:.2g}).{opposite_note}")
            phrase = (f"In the integrated HGCA atlas, sample-level abundance of "
                       f"{state} as a fraction of absorptive epithelium is **not** "
                       f"elevated in pediatric donors (<20y) relative to adults "
                       f"after adjusting for tissue and dataset.{opposite_note}")
    final_rows.append(dict(
        finding=f"BEST4 abundance ({state}) elevated in pediatric",
        gene="(abundance)",
        cell_state=state,
        direction="up" if (pd.notna(abund_row.get("beta")) and abund_row["beta"] > 0) else "down",
        beta=float(abund_row.get("beta", np.nan))
              if pd.notna(abund_row.get("beta")) else np.nan,
        q_value=float(abund_row.get("p_value", np.nan))
                 if pd.notna(abund_row.get("p_value")) else np.nan,
        n_samples_pediatric=int(abund_row.get("n_samples_pediatric", 0)),
        n_samples_adult=int(abund_row.get("n_samples_adult", 0)),
        n_datasets_pediatric=int(abund_row.get("n_datasets_pediatric", 0)),
        n_datasets_adult=int(abund_row.get("n_datasets_adult", 0)),
        lodo_sign_consistency="not_computed",
        support_level=support,
        caveat=caveat,
        suggested_paper_phrase=phrase,
    ))


final_df = pd.DataFrame(final_rows)
final_df.to_csv(P3_DIR / "age_signature_clean_summary.csv", index=False)
print("Final clean summary:")
print(final_df[["finding", "direction", "beta", "q_value",
                 "lodo_sign_consistency", "support_level"]].to_string(index=False))

## 3P. Plain-language result summary

After applying strict sample-level coverage rules (≥5 samples and ≥2 datasets per age group, ≥20 cells per sample-cellstate), and modelling `mean_log ~ pediatric_0_19 + tissue_level_1 + dataset_id` against the integrated **HGCA all-lineages** atlas:

**1. BEST4 abundance is not increased in younger donors.**
- `BEST4 Colonocytes` fraction-of-absorptive: not significantly different (p ≈ 0.87).
- `BEST4 Enterocytes` fraction-of-absorptive: nominally **decreased** in pediatric (β ≈ -0.21, p ≈ 0.047, n_pediatric = 15 / n_adult = 195). The "increased BEST4 in pediatric" hypothesis is **not supported**.

**2. Most cell-state-specific gene-level signals from the previous pass do not survive strict coverage in this atlas.**
- `DGAT1` in `BEST4 Enterocytes`, `SLC4A4` / `SLC9A2` in colonic crypt/progenitor states, and `CFTR` in `Colonocyte Progenitors` all fail the coverage rules — pediatric samples for these states come from a single dataset, so the effects cannot be separated from dataset.
- `DGAT1` in `Villus Tip Enterocytes` and `GUCA2B` in `Villus Tip Enterocytes` have OK sample coverage but the pediatric coefficient is fully absorbed by dataset fixed effects (full-fit p = 1, but LODO subsets show consistent positive effects). Treat as **unsupported by the strict full-data model** despite a directionally consistent LODO signal.

**3. The age-related signals that *do* survive concentrate in mid- and lower-villus absorptive enterocytes (small bowel).**
- `GUCA2B` **up** in pediatric `Mid Villus Enterocytes` (q ≈ 2.5e-3, β ≈ +2.26).
- `SLC4A4` **down** in pediatric `Mid Villus Enterocytes` (q ≈ 0.018, β ≈ -1.35).
- `CA7` **down** in pediatric `Mid Villus Enterocytes` (q ≈ 0.018, β ≈ -0.10).
- `GUCA2B` **up** in pediatric `Lower Villus Enterocytes` (q ≈ 0.066, β ≈ +0.86, LODO-robust across all 11 datasets).
- Borderline (q ≈ 0.066, mixed LODO): `SAR1B` and `SLC5A1` down in `Mid Villus Enterocytes`.

**4. Conservative interpretation.**
- The cleanest, dataset-independent age signal is in pediatric small-bowel **mid- and lower-villus enterocytes**: higher `GUCA2B`, lower `SLC4A4`, `CA7`, `SAR1B`, and `SLC5A1`. This is a *cell-state-specific* signature; the previously-reported `BEST4 Enterocyte` and `Crypt Top` / `Mid Crypt Colonocyte` effects do not reach this evidence bar in this atlas.
- **Do not claim this analysis explains pediatric diarrhea mortality.** It is consistent with the broader hypothesis that pediatric absorptive-epithelial maturation differs from adult, but pediatric coverage is uneven across both segments and datasets.
- **Suggested main-text role**: small supplemental or reuse-vignette panel emphasizing the mid- and lower-villus enterocyte gene-level signature and the negative-result on BEST4 abundance. The cleaner full-paper claims would require a balanced pediatric validation cohort.